# EAH stability Paper
### Yashaswini Rajendra Bhat, Kathleen Keller, Pennsylvania State University, Dept of Nutritional Sciences

#### Import required libraries and modules

In [ ]:
#import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pingouin as pg
import statsmodels.api as sm
from statsmodels.api import OLS
from statsmodels.stats.stattools import durbin_watson
import scipy.stats as stats
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from scipy.stats import shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
%matplotlib inline

In [ ]:
pd.set_option('display.max_columns', None)

#### Read in missing value imputed (output of pre-processing)

In [ ]:
df=pd.read_excel("EAH_Database_all.xlsx")

In [ ]:
df.columns

#### Figure out outliers in intake variables and replace with column mean

In [ ]:
#replacing any statistical outliers with mean values
outlier_cols=[
       'v1_meal_g', 'v1_meal_kcal', 'v1_eah_g',
       'v1_eah_kcal', 'v1_eah_sweet_g', 'v1_eah_sweet_kcal', 'v1_eah_sav_g',
       'v1_eah_sav_kcal', 'v7_meal_g', 'v7_meal_kcal', 'v7_eah_g',
       'v7_eah_kcal', 'v7_eah_sweet_g', 'v7_eah_sweet_kcal', 'v7_eah_sav_kcal',
       'v7_eah_sav_g','v1_freddy_pre_eah', 'v7_freddy_pre_eah']

In [ ]:
def replace_outliers_with_mean(df, columns):
    # Dictionary to store the count of replaced outliers for each column
    replaced_count = {}
    
    for col in columns:
        mean_val = df[col].mean()
        std_val = df[col].std()
        
        # Define bounds for outliers
        lower_bound = mean_val - (3 * std_val)
        upper_bound = mean_val + (3 * std_val)
        
        # Calculate condition for outliers
        outlier_condition = (df[col] < lower_bound) | (df[col] > upper_bound)
        
        # Count outliers that will be replaced
        replaced_count[col] = outlier_condition.sum()
        
        # Replace outliers with the mean of the series
        df[col] = df[col].mask(outlier_condition, mean_val)
        
    # Optionally, return the dictionary if you want to see the counts after function execution
    return df, replaced_count

# Usage
df, count_replaced = replace_outliers_with_mean(df, outlier_cols)
print(count_replaced)


# ICC results

In [ ]:
df_icc=df.copy()

In [ ]:
df_icc.head()

In [ ]:
# making subsets for each ICC calculation for each type of intake
EAH_kcal = df_icc[['id','v1_eah_kcal', 'v7_eah_kcal']].copy()
EAH_g = df_icc[['id','v1_eah_g', 'v7_eah_g']].copy()

meal_kcal = df_icc[['id','v1_meal_kcal', 'v7_meal_kcal']].copy()
meal_g = df_icc[['id','v1_meal_g', 'v7_meal_g']].copy()

EAH_sweet_kcal = df_icc[['id','v1_eah_sweet_kcal', 'v7_eah_sweet_kcal']].copy()
EAH_sweet_g = df_icc[['id','v1_eah_sweet_g', 'v7_eah_sweet_g']].copy()

EAH_savory_kcal = df_icc[['id','v1_eah_sav_kcal', 'v7_eah_sav_kcal']].copy()
EAH_savory_g = df_icc[['id','v1_eah_sav_g', 'v7_eah_sav_g']].copy()

In [ ]:
# converting all into long form using pd.melt
EAH_kcal_long = pd.melt(EAH_kcal, id_vars='id', value_vars=['v1_eah_kcal', 'v7_eah_kcal'],var_name='visit', value_name='intake_eah_kcal')
EAH_g_long = pd.melt(EAH_g, id_vars='id', value_vars=['v1_eah_g', 'v7_eah_g'], var_name='visit', value_name='intake_eah_g')


meal_kcal_long = pd.melt(meal_kcal, id_vars='id', value_vars=['v1_meal_kcal', 'v7_meal_kcal'], var_name='visit', value_name='intake_meal_kcal')
meal_g_long = pd.melt(meal_g, id_vars='id', value_vars=['v1_meal_g', 'v7_meal_g'], var_name='visit', value_name='intake_meal_g')

EAH_sweet_kcal_long = pd.melt(EAH_sweet_kcal, id_vars='id', value_vars=['v1_eah_sweet_kcal', 'v7_eah_sweet_kcal'], var_name='visit', value_name='intake_eah_sweet_kcal')
EAH_sweet_g_long = pd.melt(EAH_sweet_g, id_vars='id', value_vars=['v1_eah_sweet_g', 'v7_eah_sweet_g'], var_name='visit', value_name='intake_eah_sweet_g')

EAH_savory_kcal_long = pd.melt(EAH_savory_kcal, id_vars='id', value_vars=['v1_eah_sav_kcal', 'v7_eah_sav_kcal'],var_name='visit', value_name='intake_eah_sav_kcal')
EAH_savory_g_long = pd.melt(EAH_savory_g, id_vars='id', value_vars=['v1_eah_sav_g', 'v7_eah_sav_g'], var_name='visit', value_name='intake_eah_sav_g')

In [ ]:
# ICC calculation for each sub dataset
icc_result_EAH_kcal = pg.intraclass_corr(data=EAH_kcal_long, targets='id', raters='visit', ratings='intake_eah_kcal')
icc_result_EAH_g = pg.intraclass_corr(data=EAH_g_long, targets='id', raters='visit', ratings='intake_eah_g')

icc_result_meal_kcal = pg.intraclass_corr(data=meal_kcal_long, targets='id', raters='visit', ratings='intake_meal_kcal')
icc_result_meal_g = pg.intraclass_corr(data=meal_g_long, targets='id', raters='visit', ratings='intake_meal_g')

icc_result_EAH_sweet_kcal = pg.intraclass_corr(data=EAH_sweet_kcal_long, targets='id', raters='visit', ratings='intake_eah_sweet_kcal')
icc_result_EAH_sweet_g = pg.intraclass_corr(data=EAH_sweet_g_long, targets='id', raters='visit', ratings='intake_eah_sweet_g')

icc_result_EAH_savory_kcal = pg.intraclass_corr(data=EAH_savory_kcal_long, targets='id', raters='visit', ratings='intake_eah_sav_kcal')
icc_result_EAH_savory_g = pg.intraclass_corr(data=EAH_savory_g_long, targets='id', raters='visit', ratings='intake_eah_sav_g')

In [ ]:
# Create a dictionary to store the icc_result for each variable
icc_results = {
    'meal_kcal': icc_result_meal_kcal,
    'meal_g': icc_result_meal_g,
    'EAH_kcal': icc_result_EAH_kcal,
    'EAH_g': icc_result_EAH_g,
    'EAH_sweet_kcal': icc_result_EAH_sweet_kcal,
    'EAH_sweet_g': icc_result_EAH_sweet_g,
    'EAH_savory_kcal': icc_result_EAH_savory_kcal,
    'EAH_savory_g': icc_result_EAH_savory_g,
}

# List to store each row of the ICC table
icc_table_rows = []

# Loop through each variable and extract the relevant ICC2 information
for var, icc_result in icc_results.items():
    icc2_value = icc_result[icc_result['Type'] == 'ICC2']['ICC'].values[0]
    p_value = icc_result[icc_result['Type'] == 'ICC2']['pval'].values[0]
    ci95 = icc_result[icc_result['Type'] == 'ICC2']['CI95%'].values[0]

    # Append the information to the icc_table_rows list
    icc_table_rows.append({
        'Measured for': var,
        'ICC2': icc2_value,
        'p-value': p_value,
        'CI95%': ci95
    })

# Convert the list of dictionaries into a DataFrame
icc_table = pd.DataFrame(icc_table_rows)

# Print the ICC table
print(icc_table)

# Save the table to an Excel file
icc_table.to_excel('ICC results_all.xlsx', index=False)

# Spearman Rank Coefficients for confirming ICC

In [ ]:
spearman_df=df.copy()

In [ ]:
spearman_df.head()

In [ ]:
sr_cols=['v1_meal_g','v1_meal_kcal','v7_meal_g','v7_meal_kcal','v1_eah_g','v1_eah_kcal','v7_eah_g','v7_eah_kcal','v1_eah_sweet_g','v1_eah_sweet_kcal','v7_eah_sweet_g','v7_eah_sweet_kcal','v1_eah_sav_g','v1_eah_sav_kcal','v7_eah_sav_g','v7_eah_sav_kcal']

In [ ]:
# Assuming your dataframe is spearman_df and the columns are in sr_cols
spearman_results = {}

for col in sr_cols:
    if col.startswith("v1_"):
        v7_col = col.replace("v1_", "v7_")
        if v7_col in sr_cols:
            correlation, p_value = spearmanr(spearman_df[col], spearman_df[v7_col])
            formatted_correlation = "{:.2f}".format(correlation)
            formatted_p_value = "{:.2f}".format(p_value)  # Format p-value to two decimal places
            
            # Calculate the number of data points
            n = len(spearman_df)
            
            # Calculate the Fisher transformation of the correlation
            fisher_transform = 0.5 * np.log((1 + correlation) / (1 - correlation))
            
            # Calculate the standard error of the Fisher transform
            standard_error = 1 / np.sqrt(n - 3)
            
            # Calculate the margin of error for the 95% CI
            margin_of_error = 1.96 * standard_error
            
            # Calculate the lower and upper bounds of the CI
            ci_lower = np.tanh(fisher_transform - margin_of_error)
            ci_upper = np.tanh(fisher_transform + margin_of_error)
            
            formatted_ci_lower = "{:.2f}".format(ci_lower)
            formatted_ci_upper = "{:.2f}".format(ci_upper)
            
            spearman_results[col + ' & ' + v7_col] = (formatted_correlation, formatted_p_value, (formatted_ci_lower, formatted_ci_upper))

# Convert to DataFrame for better visualization
results_df = pd.DataFrame(list(spearman_results.items()), columns=['Variable Pair', ('Spearman Rank Correlation', 'p-value', '95% CI')])
print(results_df)

# Pre-processing for linear regression

### Scaling continuous values to fit regression methods next

In [ ]:
df_linear_reg=df.copy()

In [ ]:
df_linear_reg.columns

In [ ]:
scale_cols=['v1_meal_g', 'v1_meal_kcal',
       'v1_eah_g', 'v1_eah_kcal', 'v1_eah_sweet_g', 'v1_eah_sweet_kcal',
       'v1_eah_sav_g', 'v1_eah_sav_kcal', 'v7_meal_g', 'v7_meal_kcal',
       'v7_eah_g', 'v7_eah_kcal', 'v7_eah_sweet_g', 'v7_eah_sweet_kcal',
       'v7_eah_sav_kcal', 'v7_eah_sav_g','age_diff','v1_FMI','v7_FMI']

In [ ]:
# Initialize the scaler
scaler = StandardScaler()

# Scale the specified columns
scaled_values = scaler.fit_transform(df_linear_reg[scale_cols])

# Replace original columns with scaled values
df_linear_reg[scale_cols] = scaled_values

#### Median split for income and parent_ed

In [ ]:
# Recoding the income column
df_linear_reg['income'] = df_linear_reg['income'].apply(lambda x: 0 if x in [0, 1, 2] else 1)

# Recoding the parent_ed column
df_linear_reg['parent_ed'] = df_linear_reg['parent_ed'].apply(lambda x: 0 if x in [0, 1, 2] else 1)

### One-hot encode for pds tanner cat at v1 and v7 

In [ ]:
# One-hot encoding for v7_p_pds_imputed
tanner_dummies = pd.get_dummies(df_linear_reg['v7_p_pds_imputed'], prefix='v7_p_pds_imputed', drop_first=True)

# Rename the columns appropriately
tanner_dummies.columns = ['v7_p_pds_imputed_2', 'v7_p_pds_imputed_3']

# Drop the original column
df_linear_reg.drop(['v7_p_pds_imputed'], axis=1, inplace=True)

# Add the one-hot encoded columns back to the dataframe
df_linear_reg = pd.concat([df_linear_reg, tanner_dummies], axis=1)

In [ ]:
df_linear_reg.head()

In [ ]:
df_linear_reg.columns

In [ ]:
# Assuming 'bool_col1' and 'bool_col2' are your boolean columns
df_linear_reg['v7_p_pds_imputed_2'] =df_linear_reg['v7_p_pds_imputed_2'].astype(int)
df_linear_reg['v7_p_pds_imputed_3'] = df_linear_reg['v7_p_pds_imputed_3'].astype(int)


In [ ]:
# Map 'H' to 0 and 'AB' to 1 in the 'audio' column
df_linear_reg['Audio mode'] = df_linear_reg['Audio mode'].map({'H': 0, 'AB': 1})

In [ ]:
df_linear_reg.head()

# Regression Models: Intake (kcal) at visit 1 vs. FMI (adiposity) at follow-up

### Model : meal (kcal) at baseline (V1) vs V7 FMI 
##### covariates baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1, main predictor: baseline meal intake (kcal)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_mealkcalFMI = df_linear_reg[['v1_FMI','sex', 'risk_status_mom', 'income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','v1_meal_kcal']]
y_mealkcalFMI = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_mealkcalFMI = sm.add_constant(X_mealkcalFMI)

# Fit a linear regression model using ols
mod_mealkcalFMI = sm.OLS(y_mealkcalFMI, X_mealkcalFMI).fit()

# Get summary of the regression model
print(mod_mealkcalFMI.summary())


In [ ]:
import numpy as np

# Create a boolean mask for rows with NaN or inf values
mask = X_mealkcalFMI.isin([np.nan, np.inf, -np.inf]).any(axis=1)

# Filter the DataFrame to get the rows with NaN or inf values
nan_inf_rows = X_mealkcalFMI[mask]

# Get the corresponding id column values
corresponding_ids = df.loc[nan_inf_rows.index, 'id']

# Display the ids
print("IDs corresponding to rows with NaN or inf values in X_mealkcalFMI:")
print(corresponding_ids)


### Adding audio mode (human story read or read to = 1, audiobok=0)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_mealkcalFMI_audio = df_linear_reg[['v1_FMI','sex', 'risk_status_mom', 'income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','Audio mode','v1_meal_kcal']]
y_mealkcalFMI_audio = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_mealkcalFMI_audio = sm.add_constant(X_mealkcalFMI_audio)

# Fit a linear regression model using ols
mod_mealkcalFMI_audio = sm.OLS(y_mealkcalFMI_audio, X_mealkcalFMI_audio).fit()

# Get summary of the regression model
print(mod_mealkcalFMI_audio.summary())


assumption check for models

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_mealkcalFMI= mod_mealkcalFMI.resid
# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_mealkcalFMI = durbin_watson(residuals_mealkcalFMI)
print("Durbin-Watson Statistic:", dw_statistic_mealkcalFMI)
if 0 < dw_statistic_mealkcalFMI < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_mealkcalFMI.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_mealkcalFMI.columns
vif_data["VIF"] = [variance_inflation_factor(X_mealkcalFMI.values, i) for i in range(X_mealkcalFMI.shape[1])]
print(vif_data)

# 5. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_mealkcalFMI.fittedvalues, y=residuals_mealkcalFMI)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

# Breusch-Pagan test
bp_test = het_breuschpagan(residuals_mealkcalFMI, mod_mealkcalFMI.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

    
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_mealkcalFMI, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_mealkcalFMI, p_value_mealkcalFMI = shapiro(residuals_mealkcalFMI)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_mealkcalFMI)
print("p-value:", p_value_mealkcalFMI)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_mealkcalFMI > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

assumption check for audio model

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_mealkcalFMI_audio= mod_mealkcalFMI_audio.resid
# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_mealkcalFMI_audio = durbin_watson(residuals_mealkcalFMI_audio)
print("Durbin-Watson Statistic:", dw_statistic_mealkcalFMI_audio)
if 0 < dw_statistic_mealkcalFMI_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_mealkcalFMI_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_mealkcalFMI_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_mealkcalFMI_audio.values, i) for i in range(X_mealkcalFMI_audio.shape[1])]
print(vif_data)

# 5. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_mealkcalFMI_audio.fittedvalues, y=residuals_mealkcalFMI_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

# Breusch-Pagan test
bp_test = het_breuschpagan(residuals_mealkcalFMI_audio, mod_mealkcalFMI_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

    
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_mealkcalFMI_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_mealkcalFMI_audio, p_value_mealkcalFMI_audio = shapiro(residuals_mealkcalFMI_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_mealkcalFMI_audio)
print("p-value:", p_value_mealkcalFMI_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_mealkcalFMI_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### Model : EAH (kcal) at baseline (V1) vs V7 FMI
#### covariates baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1,  main predictor: baseline EAH intake (kcal)

In [ ]:
# Select specific columns as predictors (X) and response (y)

X_EAHkcalFMI = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','v1_eah_kcal']]
y_EAHkcalFMI = df_linear_reg['v7_FMI']


# Add a constant term to the predictors (intercept)
X_EAHkcalFMI = sm.add_constant(X_EAHkcalFMI)

# Fit a linear regression model using ols
mod_EAHkcalFMI = sm.OLS(y_EAHkcalFMI, X_EAHkcalFMI).fit()

# Get summary of the regression model
print(mod_EAHkcalFMI.summary())

### audio story type added

In [ ]:
# Select specific columns as predictors (X) and response (y)

X_EAHkcalFMI_audio = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','Audio mode','v1_eah_kcal']]
y_EAHkcalFMI_audio = df_linear_reg['v7_FMI']


# Add a constant term to the predictors (intercept)
X_EAHkcalFMI_audio = sm.add_constant(X_EAHkcalFMI_audio)

# Fit a linear regression model using ols
mod_EAHkcalFMI_audio = sm.OLS(y_EAHkcalFMI_audio, X_EAHkcalFMI_audio).fit()

# Get summary of the regression model
print(mod_EAHkcalFMI_audio.summary())

### adding item cleaner (0=no item cleaner, 1=item cleaner)

In [ ]:
# Select specific columns as predictors (X) and response (y)

X_EAHkcalFMI_item = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','item clean', 'v1_eah_kcal']]
y_EAHkcalFMI_item = df_linear_reg['v7_FMI']


# Add a constant term to the predictors (intercept)
X_EAHkcalFMI_item = sm.add_constant(X_EAHkcalFMI_item)

# Fit a linear regression model using ols
mod_EAHkcalFMI_item = sm.OLS(y_EAHkcalFMI_item, X_EAHkcalFMI_item).fit()

# Get summary of the regression model
print(mod_EAHkcalFMI_item.summary())

assumptions check for all three models below

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHkcalFMI = mod_EAHkcalFMI.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHkcalFMI = durbin_watson(residuals_EAHkcalFMI)
print("Durbin-Watson Statistic:", dw_statistic_EAHkcalFMI)
if 0 < dw_statistic_EAHkcalFMI < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHkcalFMI.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHkcalFMI.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHkcalFMI.values, i) for i in range(X_EAHkcalFMI.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHkcalFMI.fittedvalues, y=residuals_EAHkcalFMI)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHkcalFMI, mod_EAHkcalFMI.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")
    
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHkcalFMI, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHkcalFMI, p_value_EAHkcalFMI = shapiro(residuals_EAHkcalFMI)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHkcalFMI)
print("p-value:", p_value_EAHkcalFMI)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHkcalFMI > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHkcalFMI_audio = mod_EAHkcalFMI_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHkcalFMI_audio = durbin_watson(residuals_EAHkcalFMI_audio)
print("Durbin-Watson Statistic:", dw_statistic_EAHkcalFMI_audio)
if 0 < dw_statistic_EAHkcalFMI_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix_audio = X_EAHkcalFMI_audio.corr()
sns.heatmap(correlation_matrix_audio, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHkcalFMI_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHkcalFMI_audio.values, i) for i in range(X_EAHkcalFMI_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHkcalFMI_audio.fittedvalues, y=residuals_EAHkcalFMI_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHkcalFMI_audio, mod_EAHkcalFMI_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")
    
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHkcalFMI_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHkcalFMI_audio, p_value_EAHkcalFMI_audio = shapiro(residuals_EAHkcalFMI_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHkcalFMI_audio)
print("p-value:", p_value_EAHkcalFMI_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHkcalFMI_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHkcalFMI_item = mod_EAHkcalFMI_item.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHkcalFMI_item = durbin_watson(residuals_EAHkcalFMI_item)
print("Durbin-Watson Statistic:", dw_statistic_EAHkcalFMI_item)
if 0 < dw_statistic_EAHkcalFMI_item < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHkcalFMI_item.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHkcalFMI_item.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHkcalFMI_item.values, i) for i in range(X_EAHkcalFMI_item.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHkcalFMI_item.fittedvalues, y=residuals_EAHkcalFMI_item)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHkcalFMI_item, mod_EAHkcalFMI_item.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")
    
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHkcalFMI_item, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHkcalFMI_item, p_value_EAHkcalFMI_item = shapiro(residuals_EAHkcalFMI_item)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHkcalFMI_item)
print("p-value:", p_value_EAHkcalFMI_item)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHkcalFMI_item > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### Model : EAH Sweet (kcal) at baseline (V1) vs V7 FMI
#### covariates: baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1, main predictor: baseline EAH Sweet intake (kcal)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetkcalFMI = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff', 'v1_eah_sweet_kcal']]
y_EAHsweetkcalFMI = df_linear_reg['v7_FMI']
# Add a constant term to the predictors (intercept)
X_EAHsweetkcalFMI = sm.add_constant(X_EAHsweetkcalFMI)

# Fit a linear regression model using ols
mod_EAHsweetkcalFMI = sm.OLS(y_EAHsweetkcalFMI, X_EAHsweetkcalFMI).fit()

# Get summary of the regression model
print(mod_EAHsweetkcalFMI.summary())

#### audio type added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetkcalFMI_audio = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff', 'Audio mode', 'v1_eah_sweet_kcal']]
y_EAHsweetkcalFMI_audio = df_linear_reg['v7_FMI']
# Add a constant term to the predictors (intercept)
X_EAHsweetkcalFMI_audio = sm.add_constant(X_EAHsweetkcalFMI_audio)

# Fit a linear regression model using ols
mod_EAHsweetkcalFMI_audio = sm.OLS(y_EAHsweetkcalFMI_audio, X_EAHsweetkcalFMI_audio).fit()

# Get summary of the regression model
print(mod_EAHsweetkcalFMI_audio.summary())

#### item clean added 

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetkcalFMI_item = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','item clean','v1_eah_sweet_kcal']]
y_EAHsweetkcalFMI_item = df_linear_reg['v7_FMI']
# Add a constant term to the predictors (intercept)
X_EAHsweetkcalFMI_item = sm.add_constant(X_EAHsweetkcalFMI_item)

# Fit a linear regression model using ols
mod_EAHsweetkcalFMI_item = sm.OLS(y_EAHsweetkcalFMI_item, X_EAHsweetkcalFMI_item).fit()

# Get summary of the regression model
print(mod_EAHsweetkcalFMI_item.summary())

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsweetkcalFMI = mod_EAHsweetkcalFMI.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetkcalFMI = durbin_watson(residuals_EAHsweetkcalFMI)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetkcalFMI)
if 0 < dw_statistic_EAHsweetkcalFMI < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetkcalFMI.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetkcalFMI.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetkcalFMI.values, i) for i in range(X_EAHsweetkcalFMI.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetkcalFMI.fittedvalues, y=residuals_EAHsweetkcalFMI)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetkcalFMI,  mod_EAHsweetkcalFMI.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetkcalFMI, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetkcalFMI, p_value_EAHsweetkcalFMI = shapiro(residuals_EAHsweetkcalFMI)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetkcalFMI)
print("p-value:", p_value_EAHsweetkcalFMI)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetkcalFMI > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

model assumptiosn check for all three models

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsweetkcalFMI_audio = mod_EAHsweetkcalFMI_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetkcalFMI_audio = durbin_watson(residuals_EAHsweetkcalFMI_audio)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetkcalFMI_audio)
if 0 < dw_statistic_EAHsweetkcalFMI_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetkcalFMI_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetkcalFMI_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetkcalFMI_audio.values, i) for i in range(X_EAHsweetkcalFMI_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetkcalFMI_audio.fittedvalues, y=residuals_EAHsweetkcalFMI_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetkcalFMI_audio,  mod_EAHsweetkcalFMI_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetkcalFMI_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetkcalFMI_audio, p_value_EAHsweetkcalFMI_audio = shapiro(residuals_EAHsweetkcalFMI_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetkcalFMI_audio)
print("p-value:", p_value_EAHsweetkcalFMI_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetkcalFMI_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsweetkcalFMI_item = mod_EAHsweetkcalFMI_item.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetkcalFMI_item = durbin_watson(residuals_EAHsweetkcalFMI_item)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetkcalFMI_item)
if 0 < dw_statistic_EAHsweetkcalFMI_item < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetkcalFMI_item.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetkcalFMI_item.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetkcalFMI_item.values, i) for i in range(X_EAHsweetkcalFMI_item.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetkcalFMI_item.fittedvalues, y=residuals_EAHsweetkcalFMI_item)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetkcalFMI_item,  mod_EAHsweetkcalFMI_item.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetkcalFMI_item, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetkcalFMI_item, p_value_EAHsweetkcalFMI_item = shapiro(residuals_EAHsweetkcalFMI_item)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetkcalFMI_item)
print("p-value:", p_value_EAHsweetkcalFMI_item)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetkcalFMI_item > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

## Model : EAH savory (kcal) at baseline (V1) vs V7 FMI
#### covariates: baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1,  main predictor: baseline EAH savory intake (kcal)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavkcalFMI = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','age_diff','v1_eah_sav_kcal']]
y_EAHsavkcalFMI = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHsavkcalFMI = sm.add_constant(X_EAHsavkcalFMI)

# Fit a linear regression model using ols
mod_EAHsavkcalFMI = sm.OLS(y_EAHsavkcalFMI, X_EAHsavkcalFMI).fit()

# Get summary of the regression model
print(mod_EAHsavkcalFMI.summary())

#### audio mode added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavkcalFMI_audio = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','Audio mode','age_diff','v1_eah_sav_kcal']]
y_EAHsavkcalFMI_audio = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHsavkcalFMI_audio = sm.add_constant(X_EAHsavkcalFMI_audio)

# Fit a linear regression model using ols
mod_EAHsavkcalFMI_audio = sm.OLS(y_EAHsavkcalFMI_audio, X_EAHsavkcalFMI_audio).fit()

# Get summary of the regression model
print(mod_EAHsavkcalFMI_audio.summary())

#### added item cleaners

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavkcalFMI_item = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','item clean', 'v1_eah_sav_kcal']]
y_EAHsavkcalFMI_item = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHsavkcalFMI_item = sm.add_constant(X_EAHsavkcalFMI_item)

# Fit a linear regression model using ols
mod_EAHsavkcalFMI_item = sm.OLS(y_EAHsavkcalFMI_item, X_EAHsavkcalFMI_item).fit()

# Get summary of the regression model
print(mod_EAHsavkcalFMI_item.summary())

assumption check for all 3 models

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsavkcalFMI = mod_EAHsavkcalFMI.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavkcalFMI = durbin_watson(residuals_EAHsavkcalFMI)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavkcalFMI)
if 0 < dw_statistic_EAHsavkcalFMI < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavkcalFMI.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavkcalFMI.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavkcalFMI.values, i) for i in range(X_EAHsavkcalFMI.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavkcalFMI.fittedvalues, y=residuals_EAHsavkcalFMI)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsavkcalFMI,  mod_EAHsavkcalFMI.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavkcalFMI, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavkcalFMI, p_value_EAHsavkcalFMI = shapiro(residuals_EAHsavkcalFMI)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavkcalFMI)
print("p-value:", p_value_EAHsavkcalFMI)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsavkcalFMI > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsavkcalFMI_audio = mod_EAHsavkcalFMI_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavkcalFMI_audio = durbin_watson(residuals_EAHsavkcalFMI_audio)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavkcalFMI_audio)
if 0 < dw_statistic_EAHsavkcalFMI_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavkcalFMI_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavkcalFMI_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavkcalFMI_audio.values, i) for i in range(X_EAHsavkcalFMI_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavkcalFMI_audio.fittedvalues, y=residuals_EAHsavkcalFMI_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsavkcalFMI_audio,  mod_EAHsavkcalFMI_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavkcalFMI_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavkcalFMI_audio, p_value_EAHsavkcalFMI_audio = shapiro(residuals_EAHsavkcalFMI_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavkcalFMI_audio)
print("p-value:", p_value_EAHsavkcalFMI_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsavkcalFMI_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsavkcalFMI_item = mod_EAHsavkcalFMI_item.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavkcalFMI_item = durbin_watson(residuals_EAHsavkcalFMI_item)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavkcalFMI_item)
if 0 < dw_statistic_EAHsavkcalFMI_item < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavkcalFMI_item.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavkcalFMI_item.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavkcalFMI_item.values, i) for i in range(X_EAHsavkcalFMI_item.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavkcalFMI_item.fittedvalues, y=residuals_EAHsavkcalFMI_item)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsavkcalFMI_item,  mod_EAHsavkcalFMI_item.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavkcalFMI_item, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavkcalFMI_item, p_value_EAHsavkcalFMI_item = shapiro(residuals_EAHsavkcalFMI_item)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavkcalFMI_item)
print("p-value:", p_value_EAHsavkcalFMI_item)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsavkcalFMI_item > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

# Regression models : Baseline intake (kcal) vs follow-up weight status (BMIz)

### Model : Meal (kcal) at baseline (V1) vs V7 weight status BMIz
#### covariates : baseline BMIz, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1, main predictor: baseline meal intake (kcal)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_mealkcalBMIz = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff', 'v1_meal_kcal']]
y_mealkcalBMIz = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_mealkcalBMIz = sm.add_constant(X_mealkcalBMIz)

# Fit a linear regression model using ols
mod_mealkcalBMIz = sm.OLS(y_mealkcalBMIz, X_mealkcalBMIz).fit()

# Get summary of the regression model
print(mod_mealkcalBMIz.summary())

#### added audio mode

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_mealkcalBMIz_audio= df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','Audio mode','age_diff', 'v1_meal_kcal']]
y_mealkcalBMIz_audio = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_mealkcalBMIz_audio = sm.add_constant(X_mealkcalBMIz_audio)

# Fit a linear regression model using ols
mod_mealkcalBMIz_audio = sm.OLS(y_mealkcalBMIz_audio, X_mealkcalBMIz_audio).fit()

# Get summary of the regression model
print(mod_mealkcalBMIz_audio.summary())

assumption check for models

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_mealkcalBMIz= mod_mealkcalBMIz.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_mealkcalBMIz = durbin_watson(residuals_mealkcalBMIz)
print("Durbin-Watson Statistic:", dw_statistic_mealkcalBMIz)
if 0 < dw_statistic_mealkcalBMIz < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_mealkcalBMIz.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_mealkcalBMIz.columns
vif_data["VIF"] = [variance_inflation_factor(X_mealkcalBMIz.values, i) for i in range(X_mealkcalBMIz.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_mealkcalBMIz.fittedvalues, y=residuals_mealkcalBMIz)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_mealkcalBMIz,  mod_mealkcalBMIz.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_mealkcalBMIz, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_mealkcalBMIz, p_value_mealkcalBMIz = shapiro(residuals_mealkcalBMIz)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_mealkcalBMIz)
print("p-value:", p_value_mealkcalBMIz)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_mealkcalBMIz > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_mealkcalBMIz_audio= mod_mealkcalBMIz_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_mealkcalBMIz_audio = durbin_watson(residuals_mealkcalBMIz_audio)
print("Durbin-Watson Statistic:", dw_statistic_mealkcalBMIz_audio)
if 0 < dw_statistic_mealkcalBMIz_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_mealkcalBMIz_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_mealkcalBMIz_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_mealkcalBMIz_audio.values, i) for i in range(X_mealkcalBMIz_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_mealkcalBMIz_audio.fittedvalues, y=residuals_mealkcalBMIz_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_mealkcalBMIz_audio,  mod_mealkcalBMIz_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_mealkcalBMIz_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_mealkcalBMIz_audio, p_value_mealkcalBMIz_audio = shapiro(residuals_mealkcalBMIz_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_mealkcalBMIz_audio)
print("p-value:", p_value_mealkcalBMIz_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_mealkcalBMIz_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

## Model : EAH (kcal) at baseline (V1) vs V7 weight status BMIz
#### covariates : baseline BMIz, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1,  main predictor: baseline EAH intake (kcal)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHkcalBMIz = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff', 'v1_eah_kcal']]
y_EAHkcalBMIz = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHkcalBMIz = sm.add_constant(X_EAHkcalBMIz)

# Fit a linear regression model using ols
mod_EAHkcalBMIz = sm.OLS(y_EAHkcalBMIz, X_EAHkcalBMIz).fit()

# Get summary of the regression model
print(mod_EAHkcalBMIz.summary())

#### audio mode added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHkcalBMIz_audio = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','Audio mode', 'age_diff', 'v1_eah_kcal']]
y_EAHkcalBMIz_audio = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHkcalBMIz_audio = sm.add_constant(X_EAHkcalBMIz_audio)

# Fit a linear regression model using ols
mod_EAHkcalBMIz_audio = sm.OLS(y_EAHkcalBMIz_audio, X_EAHkcalBMIz_audio).fit()

# Get summary of the regression model
print(mod_EAHkcalBMIz_audio.summary())

#### added item clean

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHkcalBMIz_item = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','item clean','v1_eah_kcal']]
y_EAHkcalBMIz_item = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHkcalBMIz_item = sm.add_constant(X_EAHkcalBMIz_item)

# Fit a linear regression model using ols
mod_EAHkcalBMIz_item = sm.OLS(y_EAHkcalBMIz_item, X_EAHkcalBMIz_item).fit()

# Get summary of the regression model
print(mod_EAHkcalBMIz_item.summary())

assumption check for 3 models

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHkcalBMIz = mod_EAHkcalBMIz.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHkcalBMIz = durbin_watson(residuals_EAHkcalBMIz)
print("Durbin-Watson Statistic:", dw_statistic_EAHkcalBMIz)
if 0 < dw_statistic_EAHkcalBMIz < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHkcalBMIz.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHkcalBMIz.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHkcalBMIz.values, i) for i in range(X_EAHkcalBMIz.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHkcalBMIz.fittedvalues, y=residuals_EAHkcalBMIz)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHkcalBMIz,  mod_EAHkcalBMIz.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")


# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHkcalBMIz, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHkcalBMIz, p_value_EAHkcalBMIz = shapiro(residuals_EAHkcalBMIz)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHkcalBMIz)
print("p-value:", p_value_EAHkcalBMIz)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHkcalBMIz > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHkcalBMIz_audio= mod_EAHkcalBMIz_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHkcalBMIz_audio = durbin_watson(residuals_EAHkcalBMIz_audio)
print("Durbin-Watson Statistic:", dw_statistic_EAHkcalBMIz_audio)
if 0 < dw_statistic_EAHkcalBMIz_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHkcalBMIz_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHkcalBMIz_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHkcalBMIz_audio.values, i) for i in range(X_EAHkcalBMIz_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHkcalBMIz_audio.fittedvalues, y=residuals_EAHkcalBMIz_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHkcalBMIz_audio,  mod_EAHkcalBMIz_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")


# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHkcalBMIz_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHkcalBMIz_audio, p_value_EAHkcalBMIz_audio = shapiro(residuals_EAHkcalBMIz_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHkcalBMIz_audio)
print("p-value:", p_value_EAHkcalBMIz_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHkcalBMIz_audio> alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHkcalBMIz_item= mod_EAHkcalBMIz_item.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHkcalBMIz_item = durbin_watson(residuals_EAHkcalBMIz_item)
print("Durbin-Watson Statistic:", dw_statistic_EAHkcalBMIz_item)
if 0 < dw_statistic_EAHkcalBMIz_item < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHkcalBMIz_item.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHkcalBMIz_item.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHkcalBMIz_item.values, i) for i in range(X_EAHkcalBMIz_item.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHkcalBMIz_item.fittedvalues, y=residuals_EAHkcalBMIz_item)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHkcalBMIz_item,  mod_EAHkcalBMIz_item.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")


# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHkcalBMIz_item, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHkcalBMIz_item, p_value_EAHkcalBMIz_item = shapiro(residuals_EAHkcalBMIz_item)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHkcalBMIz_item)
print("p-value:", p_value_EAHkcalBMIz_item)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHkcalBMIz_item > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

## Model : EAH sweet (kcal) at baseline (V1) vs V7 weight status BMIz
#### covariates : baseline BMIz, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1,  main predictor: baseline EAH sweet intake (kcal)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetkcalBMIz = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','v1_eah_sweet_kcal']]
y_EAHsweetkcalBMIz = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHsweetkcalBMIz = sm.add_constant(X_EAHsweetkcalBMIz)

# Fit a linear regression model using ols
mod_EAHsweetkcalBMIz= sm.OLS(y_EAHsweetkcalBMIz, X_EAHsweetkcalBMIz).fit()

# Get summary of the regression model
print(mod_EAHsweetkcalBMIz.summary())

### audio added 

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetkcalBMIz_audio = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','Audio mode', 'age_diff','v1_eah_sweet_kcal']]
y_EAHsweetkcalBMIz_audio= df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHsweetkcalBMIz_audio = sm.add_constant(X_EAHsweetkcalBMIz_audio)

# Fit a linear regression model using ols
mod_EAHsweetkcalBMIz_audio= sm.OLS(y_EAHsweetkcalBMIz_audio, X_EAHsweetkcalBMIz_audio).fit()

# Get summary of the regression model
print(mod_EAHsweetkcalBMIz_audio.summary())

#### item clean added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetkcalBMIz_item = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff', 'item clean', 'v1_eah_sweet_kcal']]
y_EAHsweetkcalBMIz_item = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHsweetkcalBMIz_item = sm.add_constant(X_EAHsweetkcalBMIz_item)

# Fit a linear regression model using ols
mod_EAHsweetkcalBMIz_item = sm.OLS(y_EAHsweetkcalBMIz_item, X_EAHsweetkcalBMIz_item).fit()

# Get summary of the regression model
print(mod_EAHsweetkcalBMIz_item.summary())

assumption check for all 3 models

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsweetkcalBMIz= mod_EAHsweetkcalBMIz.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetkcalBMIz = durbin_watson(residuals_EAHsweetkcalBMIz)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetkcalBMIz)
if 0 < dw_statistic_EAHsweetkcalBMIz < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetkcalBMIz.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetkcalBMIz.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetkcalBMIz.values, i) for i in range(X_EAHsweetkcalBMIz.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetkcalBMIz.fittedvalues, y=residuals_EAHsweetkcalBMIz)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetkcalBMIz,  mod_EAHsweetkcalBMIz.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetkcalBMIz, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetkcalBMIz, p_value_EAHsweetkcalBMIz = shapiro(residuals_EAHsweetkcalBMIz)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetkcalBMIz)
print("p-value:", p_value_EAHsweetkcalBMIz)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetkcalBMIz > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsweetkcalBMIz_audio= mod_EAHsweetkcalBMIz_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetkcalBMIz_audio = durbin_watson(residuals_EAHsweetkcalBMIz_audio)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetkcalBMIz_audio)
if 0 < dw_statistic_EAHsweetkcalBMIz_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetkcalBMIz_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetkcalBMIz_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetkcalBMIz_audio.values, i) for i in range(X_EAHsweetkcalBMIz_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetkcalBMIz_audio.fittedvalues, y=residuals_EAHsweetkcalBMIz_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetkcalBMIz_audio,  mod_EAHsweetkcalBMIz_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetkcalBMIz_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetkcalBMIz_audio, p_value_EAHsweetkcalBMIz_audio = shapiro(residuals_EAHsweetkcalBMIz_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetkcalBMIz_audio)
print("p-value:", p_value_EAHsweetkcalBMIz_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetkcalBMIz_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsweetkcalBMIz_item = mod_EAHsweetkcalBMIz_item.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetkcalBMIz_item = durbin_watson(residuals_EAHsweetkcalBMIz_item)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetkcalBMIz_item)
if 0 < dw_statistic_EAHsweetkcalBMIz_item < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetkcalBMIz_item.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetkcalBMIz_item.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetkcalBMIz_item.values, i) for i in range(X_EAHsweetkcalBMIz_item.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetkcalBMIz_item.fittedvalues, y=residuals_EAHsweetkcalBMIz_item)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetkcalBMIz_item,  mod_EAHsweetkcalBMIz_item.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetkcalBMIz_item, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetkcalBMIz_item, p_value_EAHsweetkcalBMIz_item = shapiro(residuals_EAHsweetkcalBMIz_item)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetkcalBMIz_item)
print("p-value:", p_value_EAHsweetkcalBMIz_item)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetkcalBMIz_item > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

## Model : EAH savory (kcal) at baseline (V1) vs V7 weight status BMIz
#### covariates :baseline BMIz, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1, main predictor: baseline EAH savory intake (kcal)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavkcalBMIz = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff', 'v1_eah_sav_kcal']]
y_EAHsavkcalBMIz = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHsavkcalBMIz = sm.add_constant(X_EAHsavkcalBMIz)

# Fit a linear regression model using ols
mod_EAHsavkcalBMIz= sm.OLS(y_EAHsavkcalBMIz, X_EAHsavkcalBMIz).fit()

# Get summary of the regression model
print(mod_EAHsavkcalBMIz.summary())

#### added audio mode

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavkcalBMIz_audio = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','Audio mode','age_diff', 'v1_eah_sav_kcal']]
y_EAHsavkcalBMIz_audio = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHsavkcalBMIz_audio = sm.add_constant(X_EAHsavkcalBMIz_audio)

# Fit a linear regression model using ols
mod_EAHsavkcalBMIz_audio= sm.OLS(y_EAHsavkcalBMIz_audio, X_EAHsavkcalBMIz_audio).fit()

# Get summary of the regression model
print(mod_EAHsavkcalBMIz_audio.summary())

#### added item clean

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavkcalBMIz_item = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','item clean','v1_eah_sav_kcal']]
y_EAHsavkcalBMIz_item = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHsavkcalBMIz_item = sm.add_constant(X_EAHsavkcalBMIz_item)

# Fit a linear regression model using ols
mod_EAHsavkcalBMIz_item = sm.OLS(y_EAHsavkcalBMIz_item, X_EAHsavkcalBMIz_item).fit()

# Get summary of the regression model
print(mod_EAHsavkcalBMIz_item.summary())

assumption check for 3 models

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsavkcalBMIz= mod_EAHsavkcalBMIz.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavkcalBMIz = durbin_watson(residuals_EAHsavkcalBMIz)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavkcalBMIz)
if 0 < dw_statistic_EAHsavkcalBMIz < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavkcalBMIz.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavkcalBMIz.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavkcalBMIz.values, i) for i in range(X_EAHsavkcalBMIz.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavkcalBMIz.fittedvalues, y=residuals_EAHsavkcalBMIz)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsavkcalBMIz,  mod_EAHsavkcalBMIz.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")
    
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavkcalBMIz, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavkcalBMIz, p_value_EAHsavkcalBMIz = shapiro(residuals_EAHsavkcalBMIz)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavkcalBMIz)
print("p-value:", p_value_EAHsavkcalBMIz)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsavkcalBMIz > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsavkcalBMIz_audio= mod_EAHsavkcalBMIz_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavkcalBMIz_audio = durbin_watson(residuals_EAHsavkcalBMIz_audio)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavkcalBMIz_audio)
if 0 < dw_statistic_EAHsavkcalBMIz_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavkcalBMIz_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavkcalBMIz_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavkcalBMIz_audio.values, i) for i in range(X_EAHsavkcalBMIz_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavkcalBMIz_audio.fittedvalues, y=residuals_EAHsavkcalBMIz_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsavkcalBMIz_audio,  mod_EAHsavkcalBMIz_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")
    
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavkcalBMIz_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavkcalBMIz_audio, p_value_EAHsavkcalBMIz_audio = shapiro(residuals_EAHsavkcalBMIz_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavkcalBMIz_audio)
print("p-value:", p_value_EAHsavkcalBMIz_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsavkcalBMIz_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsavkcalBMIz_item= mod_EAHsavkcalBMIz_item.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavkcalBMIz_item = durbin_watson(residuals_EAHsavkcalBMIz_item)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavkcalBMIz_item)
if 0 < dw_statistic_EAHsavkcalBMIz_item < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavkcalBMIz_item.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavkcalBMIz_item.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavkcalBMIz_item.values, i) for i in range(X_EAHsavkcalBMIz_item.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavkcalBMIz_item.fittedvalues, y=residuals_EAHsavkcalBMIz_item)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsavkcalBMIz_item,  mod_EAHsavkcalBMIz_item.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")
    
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavkcalBMIz_item, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavkcalBMIz_item, p_value_EAHsavkcalBMIz_item = shapiro(residuals_EAHsavkcalBMIz_item)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavkcalBMIz_item)
print("p-value:", p_value_EAHsavkcalBMIz_item)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsavkcalBMIz_item > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

## Regression model: Intake (gram) at baseline V1 vs follow-up adiposity (FMI)

### Model : Meal (gram) at baseline (V1) vs V7 follow-up adiposity FMI
#### covariates : baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1, main predictor: baseline meal intake (gram)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_mealgFMI = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff', 'v1_meal_g']]
y_mealgFMI = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_mealgFMI = sm.add_constant(X_mealgFMI)

# Fit a linear regression model using ols
mod_mealgFMI= sm.OLS(y_mealgFMI, X_mealgFMI).fit()

# Get summary of the regression model
print(mod_mealgFMI.summary())

audio added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_mealgFMI_audio = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','Audio mode','age_diff', 'v1_meal_g']]
y_mealgFMI_audio = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_mealgFMI_audio = sm.add_constant(X_mealgFMI_audio)

# Fit a linear regression model using ols
mod_mealgFMI_audio= sm.OLS(y_mealgFMI_audio, X_mealgFMI_audio).fit()

# Get summary of the regression model
print(mod_mealgFMI_audio.summary())

assumptions check

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_mealgFMI= mod_mealgFMI.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_mealgFMI = durbin_watson(residuals_mealgFMI)
print("Durbin-Watson Statistic:", dw_statistic_mealgFMI)
if 0 < dw_statistic_mealgFMI < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_mealgFMI.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_mealgFMI.columns
vif_data["VIF"] = [variance_inflation_factor(X_mealgFMI.values, i) for i in range(X_mealgFMI.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_mealgFMI.fittedvalues, y=residuals_mealgFMI)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_mealgFMI,  mod_mealgFMI.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_mealgFMI, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_mealgFMI, p_value_mealgFMI = shapiro(residuals_mealgFMI)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_mealgFMI)
print("p-value:", p_value_mealgFMI)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_mealgFMI > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_mealgFMI_audio= mod_mealgFMI_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_mealgFMI_audio = durbin_watson(residuals_mealgFMI_audio)
print("Durbin-Watson Statistic:", dw_statistic_mealgFMI_audio)
if 0 < dw_statistic_mealgFMI_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_mealgFMI_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_mealgFMI_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_mealgFMI_audio.values, i) for i in range(X_mealgFMI_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_mealgFMI_audio.fittedvalues, y=residuals_mealgFMI_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_mealgFMI_audio,  mod_mealgFMI_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_mealgFMI_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_mealgFMI_audio, p_value_mealgFMI_audio = shapiro(residuals_mealgFMI_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_mealgFMI_audio)
print("p-value:", p_value_mealgFMI_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_mealgFMI_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### Model : EAH (gram) at baseline (V1) vs V7 follow-up adiposity FMI (v7)
#### covariates :baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1,  main predictor: baseline EAH intake (gram)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHgFMI = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff','v1_eah_g']]
y_EAHgFMI = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHgFMI = sm.add_constant(X_EAHgFMI)

# Fit a linear regression model using ols
mod_EAHgFMI= sm.OLS(y_EAHgFMI, X_EAHgFMI).fit()

# Get summary of the regression model
print(mod_EAHgFMI.summary())

audio mode added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHgFMI_audio= df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'Audio mode','age_diff','v1_eah_g']]
y_EAHgFMI_audio = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHgFMI_audio = sm.add_constant(X_EAHgFMI_audio)

# Fit a linear regression model using ols
mod_EAHgFMI_audio= sm.OLS(y_EAHgFMI_audio, X_EAHgFMI_audio).fit()

# Get summary of the regression model
print(mod_EAHgFMI_audio.summary())

item cleaner added as covariate

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHgFMI_item= df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff','item clean','v1_eah_g']]
y_EAHgFMI_item = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHgFMI_item = sm.add_constant(X_EAHgFMI_item)

# Fit a linear regression model using ols
mod_EAHgFMI_item = sm.OLS(y_EAHgFMI_item, X_EAHgFMI_item).fit()

# Get summary of the regression model
print(mod_EAHgFMI_item.summary())

assumption checked

In [ ]:
# checking for model assumptions 
# Calculate the residuals
residuals_EAHgFMI= mod_EAHgFMI.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHgFMI = durbin_watson(residuals_EAHgFMI)
print("Durbin-Watson Statistic:", dw_statistic_EAHgFMI)
if 0 < dw_statistic_EAHgFMI < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHgFMI.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHgFMI.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHgFMI.values, i) for i in range(X_EAHgFMI.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHgFMI.fittedvalues, y=residuals_EAHgFMI)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHgFMI,  mod_EAHgFMI.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")


# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHgFMI, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHgFMI, p_value_EAHgFMI = shapiro(residuals_EAHgFMI)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHgFMI)
print("p-value:", p_value_EAHgFMI)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHgFMI > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions 
# Calculate the residuals
residuals_EAHgFMI_audio= mod_EAHgFMI_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHgFMI_audio = durbin_watson(residuals_EAHgFMI_audio)
print("Durbin-Watson Statistic:", dw_statistic_EAHgFMI_audio)
if 0 < dw_statistic_EAHgFMI_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHgFMI_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHgFMI_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHgFMI_audio.values, i) for i in range(X_EAHgFMI_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHgFMI_audio.fittedvalues, y=residuals_EAHgFMI_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHgFMI_audio,  mod_EAHgFMI_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")


# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHgFMI_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHgFMI_audio, p_value_EAHgFMI_audio = shapiro(residuals_EAHgFMI_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHgFMI_audio)
print("p-value:", p_value_EAHgFMI_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHgFMI_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions 
# Calculate the residuals
residuals_EAHgFMI_item = mod_EAHgFMI_item.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHgFMI_item = durbin_watson(residuals_EAHgFMI_item)
print("Durbin-Watson Statistic:", dw_statistic_EAHgFMI_item)
if 0 < dw_statistic_EAHgFMI_item < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHgFMI_item.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHgFMI_item.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHgFMI_item.values, i) for i in range(X_EAHgFMI_item.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHgFMI_item.fittedvalues, y=residuals_EAHgFMI_item)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHgFMI_item,  mod_EAHgFMI_item.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")


# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHgFMI_item, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHgFMI_item, p_value_EAHgFMI_item = shapiro(residuals_EAHgFMI_item)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHgFMI_item)
print("p-value:", p_value_EAHgFMI_item)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHgFMI_item > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### Model : EAH sweet (gram) at baseline (V1) vs V7 follow-up adiposity FMI (v7)
#### covariates : baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1,  main predictor: baseline EAH sweet intake (gram)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetgFMI = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','v1_eah_sweet_g']]
y_EAHsweetgFMI = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHsweetgFMI = sm.add_constant(X_EAHsweetgFMI)

# Fit a linear regression model using ols
mod_EAHsweetgFMI= sm.OLS(y_EAHsweetgFMI, X_EAHsweetgFMI).fit()

# Get summary of the regression model
print(mod_EAHsweetgFMI.summary())

added audio mode

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetgFMI_audio = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','Audio mode','age_diff','v1_eah_sweet_g']]
y_EAHsweetgFMI_audio = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHsweetgFMI_audio = sm.add_constant(X_EAHsweetgFMI_audio)

# Fit a linear regression model using ols
mod_EAHsweetgFMI_audio= sm.OLS(y_EAHsweetgFMI_audio, X_EAHsweetgFMI_audio).fit()

# Get summary of the regression model
print(mod_EAHsweetgFMI_audio.summary())

added item cleaner

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetgFMI_item = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','item clean', 'v1_eah_sweet_g']]
y_EAHsweetgFMI_item = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHsweetgFMI_item = sm.add_constant(X_EAHsweetgFMI_item)

# Fit a linear regression model using ols
mod_EAHsweetgFMI_item = sm.OLS(y_EAHsweetgFMI_item, X_EAHsweetgFMI_item).fit()

# Get summary of the regression model
print(mod_EAHsweetgFMI_item.summary())

assumption check

In [ ]:
# checking for model assumptions
# Calculate the residuals
from scipy.stats import shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
residuals_EAHsweetgFMI= mod_EAHsweetgFMI.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetgFMI = durbin_watson(residuals_EAHsweetgFMI)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetgFMI)
if 0 < dw_statistic_EAHsweetgFMI < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetgFMI.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetgFMI.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetgFMI.values, i) for i in range(X_EAHsweetgFMI.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetgFMI.fittedvalues, y=residuals_EAHsweetgFMI)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetgFMI,  mod_EAHsweetgFMI.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")   
    
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetgFMI, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetgFMI, p_value_EAHsweetgFMI = shapiro(residuals_EAHsweetgFMI)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetgFMI)
print("p-value:", p_value_EAHsweetgFMI)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetgFMI > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
from scipy.stats import shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
residuals_EAHsweetgFMI_audio= mod_EAHsweetgFMI_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetgFMI_audio = durbin_watson(residuals_EAHsweetgFMI_audio)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetgFMI_audio)
if 0 < dw_statistic_EAHsweetgFMI_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetgFMI_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetgFMI_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetgFMI_audio.values, i) for i in range(X_EAHsweetgFMI_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetgFMI_audio.fittedvalues, y=residuals_EAHsweetgFMI_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetgFMI_audio,  mod_EAHsweetgFMI_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")   
    
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetgFMI_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetgFMI_audio, p_value_EAHsweetgFMI_audio = shapiro(residuals_EAHsweetgFMI_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetgFMI_audio)
print("p-value:", p_value_EAHsweetgFMI_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetgFMI_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
from scipy.stats import shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
residuals_EAHsweetgFMI_item = mod_EAHsweetgFMI_item.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetgFMI_item = durbin_watson(residuals_EAHsweetgFMI_item)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetgFMI_item)
if 0 < dw_statistic_EAHsweetgFMI_item < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetgFMI_item.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetgFMI_item.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetgFMI_item.values, i) for i in range(X_EAHsweetgFMI_item.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetgFMI_item.fittedvalues, y=residuals_EAHsweetgFMI_item)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetgFMI_item,  mod_EAHsweetgFMI_item.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")   
    
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetgFMI_item, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetgFMI_item, p_value_EAHsweetgFMI_item = shapiro(residuals_EAHsweetgFMI_item)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetgFMI_item)
print("p-value:", p_value_EAHsweetgFMI_item)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetgFMI_item > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### Model : EAH savory (gram) at baseline (V1) vs V7 follow-up adiposity FMI (v7)
#### covariates : baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1,  main predictor: baseline EAH savory intake (gram)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavgFMI = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','v1_eah_sav_g']]
y_EAHsavgFMI = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHsavgFMI = sm.add_constant(X_EAHsavgFMI)

# Fit a linear regression model using ols
mod_EAHsavgFMI= sm.OLS(y_EAHsavgFMI, X_EAHsavgFMI).fit()

# Get summary of the regression model
print(mod_EAHsavgFMI.summary())

audio mode added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavgFMI_audio = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','Audio mode','v1_eah_sav_g']]
y_EAHsavgFMI_audio = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHsavgFMI_audio = sm.add_constant(X_EAHsavgFMI_audio)

# Fit a linear regression model using ols
mod_EAHsavgFMI_audio= sm.OLS(y_EAHsavgFMI_audio, X_EAHsavgFMI_audio).fit()

# Get summary of the regression model
print(mod_EAHsavgFMI_audio.summary())

item cleaner added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavgFMI_item = df_linear_reg[['v1_FMI','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','item clean','v1_eah_sav_g']]
y_EAHsavgFMI_item = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHsavgFMI_item = sm.add_constant(X_EAHsavgFMI_item)

# Fit a linear regression model using ols
mod_EAHsavgFMI_item = sm.OLS(y_EAHsavgFMI_item, X_EAHsavgFMI_item).fit()

# Get summary of the regression model
print(mod_EAHsavgFMI_item.summary())

assumption check for models

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsavgFMI= mod_EAHsavgFMI.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavgFMI = durbin_watson(residuals_EAHsavgFMI)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavgFMI)
if 0 < dw_statistic_EAHsavgFMI < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavgFMI.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavgFMI.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavgFMI.values, i) for i in range(X_EAHsavgFMI.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavgFMI.fittedvalues, y=residuals_EAHsavgFMI)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsavgFMI,  mod_EAHsavgFMI.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")  

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavgFMI, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavgFMI, p_value_EAHsavgFMI = shapiro(residuals_EAHsavgFMI)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavgFMI)
print("p-value:", p_value_EAHsavgFMI)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level 
if p_value_EAHsavgFMI > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsavgFMI_audio= mod_EAHsavgFMI_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavgFMI_audio = durbin_watson(residuals_EAHsavgFMI_audio)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavgFMI_audio)
if 0 < dw_statistic_EAHsavgFMI_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavgFMI_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavgFMI_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavgFMI_audio.values, i) for i in range(X_EAHsavgFMI_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavgFMI_audio.fittedvalues, y=residuals_EAHsavgFMI_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsavgFMI_audio,  mod_EAHsavgFMI_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")  

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavgFMI_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavgFMI_audio, p_value_EAHsavgFMI_audio = shapiro(residuals_EAHsavgFMI_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavgFMI_audio)
print("p-value:", p_value_EAHsavgFMI_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level 
if p_value_EAHsavgFMI_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsavgFMI_item = mod_EAHsavgFMI_item.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavgFMI_item = durbin_watson(residuals_EAHsavgFMI_item)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavgFMI_item)
if 0 < dw_statistic_EAHsavgFMI_item < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavgFMI_item.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavgFMI_item.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavgFMI_item.values, i) for i in range(X_EAHsavgFMI_item.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavgFMI_item.fittedvalues, y=residuals_EAHsavgFMI_item)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsavgFMI_item,  mod_EAHsavgFMI_item.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")  

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavgFMI_item, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavgFMI_item, p_value_EAHsavgFMI_item = shapiro(residuals_EAHsavgFMI_item)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavgFMI_item)
print("p-value:", p_value_EAHsavgFMI_item)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level 
if p_value_EAHsavgFMI_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

## Regression models: Intake (gram) at baseline (v1) vs follow-up (v7) weight status (BMIz)

### Model : Meal (gram) at baseline (V1) vs V7 follow-up weight status BMIz (v7)
#### covariates : baseline BMIz, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1,  main predictor: baseline meal intake (gram)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_mealgBMIz = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff','v1_meal_g']]
y_mealgBMIz = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_mealgBMIz= sm.add_constant(X_mealgBMIz)

# Fit a linear regression model using ols
mod_mealgBMIz= sm.OLS(y_mealgBMIz, X_mealgBMIz).fit()

# Get summary of the regression model
print(mod_mealgBMIz.summary())

audio mode added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_mealgBMIz_audio = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','Audio mode','age_diff','v1_meal_g']]
y_mealgBMIz_audio = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_mealgBMIz_audio= sm.add_constant(X_mealgBMIz_audio)

# Fit a linear regression model using ols
mod_mealgBMIz_audio= sm.OLS(y_mealgBMIz_audio, X_mealgBMIz_audio).fit()

# Get summary of the regression model
print(mod_mealgBMIz_audio.summary())

assumption check for models

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_mealgBMIz= mod_mealgBMIz.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_mealgBMIz = durbin_watson(residuals_mealgBMIz)
print("Durbin-Watson Statistic:", dw_statistic_mealgBMIz)
if 0 < dw_statistic_mealgBMIz < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_mealgBMIz.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_mealgBMIz.columns
vif_data["VIF"] = [variance_inflation_factor(X_mealgBMIz.values, i) for i in range(X_mealgBMIz.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_mealgBMIz.fittedvalues, y=residuals_mealgBMIz)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_mealgBMIz,  mod_mealgBMIz.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")  
      
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_mealgBMIz, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_mealgBMIz, p_value_mealgBMIz = shapiro(residuals_mealgBMIz)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_mealgBMIz)
print("p-value:", p_value_mealgBMIz)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_mealgBMIz > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_mealgBMIz_audio= mod_mealgBMIz_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_mealgBMIz_audio = durbin_watson(residuals_mealgBMIz_audio)
print("Durbin-Watson Statistic:", dw_statistic_mealgBMIz_audio)
if 0 < dw_statistic_mealgBMIz_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_mealgBMIz_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_mealgBMIz_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_mealgBMIz_audio.values, i) for i in range(X_mealgBMIz_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_mealgBMIz_audio.fittedvalues, y=residuals_mealgBMIz_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_mealgBMIz_audio,  mod_mealgBMIz_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")  
      
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_mealgBMIz_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_mealgBMIz_audio, p_value_mealgBMIz_audio = shapiro(residuals_mealgBMIz_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_mealgBMIz_audio)
print("p-value:", p_value_mealgBMIz_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_mealgBMIz_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### Model : EAH (gram) at baseline (V1) vs V7 follow-up weight status BMIz (v7)
#### covariates : baseline BMIz, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1,  main predictor: baseline EAH intake (gram)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHgBMIz = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff', 'v1_eah_g']]
y_EAHgBMIz = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHgBMIz= sm.add_constant(X_EAHgBMIz)

# Fit a linear regression model using ols
mod_EAHgBMIz= sm.OLS(y_EAHgBMIz, X_EAHgBMIz).fit()

# Get summary of the regression model
print(mod_EAHgBMIz.summary())

audio mode added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHgBMIz_audio = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'Audio mode', 'age_diff', 'v1_eah_g']]
y_EAHgBMIz_audio = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHgBMIz_audio= sm.add_constant(X_EAHgBMIz_audio)

# Fit a linear regression model using ols
mod_EAHgBMIz_audio= sm.OLS(y_EAHgBMIz_audio, X_EAHgBMIz_audio).fit()

# Get summary of the regression model
print(mod_EAHgBMIz_audio.summary())

item clean added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHgBMIz_item = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff', 'item clean', 'v1_eah_g']]
y_EAHgBMIz_item = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHgBMIz_item = sm.add_constant(X_EAHgBMIz_item)

# Fit a linear regression model using ols
mod_EAHgBMIz_item = sm.OLS(y_EAHgBMIz_item, X_EAHgBMIz_item).fit()

# Get summary of the regression model
print(mod_EAHgBMIz_item.summary())

assumption check for models

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHgBMIz= mod_EAHgBMIz.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHgBMIz = durbin_watson(residuals_EAHgBMIz)
print("Durbin-Watson Statistic:", dw_statistic_EAHgBMIz)
if 0 < dw_statistic_EAHgBMIz < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHgBMIz.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHgBMIz.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHgBMIz.values, i) for i in range(X_EAHgBMIz.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHgBMIz.fittedvalues, y=residuals_EAHgBMIz)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHgBMIz,  mod_EAHgBMIz.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")  
      
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHgBMIz, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHgBMIz, p_value_EAHgBMIz = shapiro(residuals_EAHgBMIz)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHgBMIz)
print("p-value:", p_value_EAHgBMIz)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHgBMIz > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHgBMIz_audio= mod_EAHgBMIz_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHgBMIz_audio = durbin_watson(residuals_EAHgBMIz_audio)
print("Durbin-Watson Statistic:", dw_statistic_EAHgBMIz_audio)
if 0 < dw_statistic_EAHgBMIz_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHgBMIz_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHgBMIz_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHgBMIz_audio.values, i) for i in range(X_EAHgBMIz_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHgBMIz_audio.fittedvalues, y=residuals_EAHgBMIz)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHgBMIz_audio,  mod_EAHgBMIz_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")  
      
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHgBMIz_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHgBMIz_audio, p_value_EAHgBMIz_audio = shapiro(residuals_EAHgBMIz_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHgBMIz_audio)
print("p-value:", p_value_EAHgBMIz_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHgBMIz_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHgBMIz_item = mod_EAHgBMIz_item.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHgBMIz_item = durbin_watson(residuals_EAHgBMIz_item)
print("Durbin-Watson Statistic:", dw_statistic_EAHgBMIz_item)
if 0 < dw_statistic_EAHgBMIz_item < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHgBMIz_item.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHgBMIz_item.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHgBMIz_item.values, i) for i in range(X_EAHgBMIz_item.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHgBMIz_item.fittedvalues, y=residuals_EAHgBMIz)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHgBMIz_item,  mod_EAHgBMIz_item.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")  
      
# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHgBMIz_item, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHgBMIz_item, p_value_EAHgBMIz_item = shapiro(residuals_EAHgBMIz_item)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHgBMIz_item)
print("p-value:", p_value_EAHgBMIz_item)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHgBMIz_item > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### Model : EAH sweet (gram) at baseline (V1) vs V7 follow-up weight status BMIz (v7)
#### covariates :baseline BMIz, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1,  main predictor: baseline EAH sweet intake (gram)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetgBMIz = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff', 'v1_eah_sweet_g']]
y_EAHsweetgBMIz = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHsweetgBMIz= sm.add_constant(X_EAHsweetgBMIz)

# Fit a linear regression model using ols
mod_EAHsweetgBMIz= sm.OLS(y_EAHsweetgBMIz, X_EAHsweetgBMIz).fit()

# Get summary of the regression model
print(mod_EAHsweetgBMIz.summary())

audio mode added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetgBMIz_audio = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'Audio mode', 'age_diff', 'v1_eah_sweet_g']]
y_EAHsweetgBMIz_audio = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHsweetgBMIz_audio= sm.add_constant(X_EAHsweetgBMIz_audio)

# Fit a linear regression model using ols
mod_EAHsweetgBMIz_audio= sm.OLS(y_EAHsweetgBMIz_audio, X_EAHsweetgBMIz_audio).fit()

# Get summary of the regression model
print(mod_EAHsweetgBMIz_audio.summary())

item cleaner added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetgBMIz_item = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff','item clean','v1_eah_sweet_g']]
y_EAHsweetgBMIz_item = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHsweetgBMIz_item = sm.add_constant(X_EAHsweetgBMIz_item)

# Fit a linear regression model using ols
mod_EAHsweetgBMIz_item = sm.OLS(y_EAHsweetgBMIz_item, X_EAHsweetgBMIz_item).fit()

# Get summary of the regression model
print(mod_EAHsweetgBMIz_item.summary())

model assumption check 

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsweetgBMIz= mod_EAHsweetgBMIz.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetgBMIz = durbin_watson(residuals_EAHsweetgBMIz)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetgBMIz)
if 0 < dw_statistic_EAHsweetgBMIz < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetgBMIz.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetgBMIz.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetgBMIz.values, i) for i in range(X_EAHsweetgBMIz.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetgBMIz.fittedvalues, y=residuals_EAHsweetgBMIz)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetgBMIz,  mod_EAHsweetgBMIz.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")  
      

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetgBMIz, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetgBMIz, p_value_EAHsweetgBMIz = shapiro(residuals_EAHsweetgBMIz)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetgBMIz)
print("p-value:", p_value_EAHsweetgBMIz)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetgBMIz > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsweetgBMIz_audio= mod_EAHsweetgBMIz_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetgBMIz_audio = durbin_watson(residuals_EAHsweetgBMIz_audio)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetgBMIz_audio)
if 0 < dw_statistic_EAHsweetgBMIz_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetgBMIz_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetgBMIz_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetgBMIz_audio.values, i) for i in range(X_EAHsweetgBMIz_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetgBMIz_audio.fittedvalues, y=residuals_EAHsweetgBMIz_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetgBMIz_audio,  mod_EAHsweetgBMIz_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")  
      

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetgBMIz_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetgBMIz_audio, p_value_EAHsweetgBMIz_audio = shapiro(residuals_EAHsweetgBMIz_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetgBMIz_audio)
print("p-value:", p_value_EAHsweetgBMIz_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetgBMIz_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsweetgBMIz_item = mod_EAHsweetgBMIz_item.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetgBMIz_item = durbin_watson(residuals_EAHsweetgBMIz_item)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetgBMIz_item)
if 0 < dw_statistic_EAHsweetgBMIz_item < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetgBMIz_item.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetgBMIz_item.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetgBMIz_item.values, i) for i in range(X_EAHsweetgBMIz_item.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetgBMIz_item.fittedvalues, y=residuals_EAHsweetgBMIz_item)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetgBMIz_item,  mod_EAHsweetgBMIz_item.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)")  
      

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetgBMIz_item, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetgBMIz_item, p_value_EAHsweetgBMIz_item = shapiro(residuals_EAHsweetgBMIz_item)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetgBMIz_item)
print("p-value:", p_value_EAHsweetgBMIz_item)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetgBMIz_item > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### Model : EAH savory (gram) at baseline (V1) vs V7 follow-up weight status BMIz (v7)
#### covariates : baseline BMIz, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1,  main predictor: baseline EAH savory intake (gram)

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavgBMIz = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff', 'v1_eah_sav_g']]
y_EAHsavgBMIz = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHsavgBMIz= sm.add_constant(X_EAHsavgBMIz)

# Fit a linear regression model using ols
mod_EAHsavgBMIz= sm.OLS(y_EAHsavgBMIz, X_EAHsavgBMIz).fit()

# Get summary of the regression model
print(mod_EAHsavgBMIz.summary())

audio mode added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavgBMIz_audio = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3',  'Audio mode', 'age_diff', 'v1_eah_sav_g']]
y_EAHsavgBMIz_audio = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHsavgBMIz_audio = sm.add_constant(X_EAHsavgBMIz_audio)

# Fit a linear regression model using ols
mod_EAHsavgBMIz_audio = sm.OLS(y_EAHsavgBMIz_audio, X_EAHsavgBMIz_audio).fit()

# Get summary of the regression model
print(mod_EAHsavgBMIz_audio.summary())

item clean added

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavgBMIz_item = df_linear_reg[['bmi_z','sex', 'risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff', 'item clean','v1_eah_sav_g']]
y_EAHsavgBMIz_item = df_linear_reg['v7_bmi_z']

# Add a constant term to the predictors (intercept)
X_EAHsavgBMIz_item = sm.add_constant(X_EAHsavgBMIz_item)

# Fit a linear regression model using ols
mod_EAHsavgBMIz_item = sm.OLS(y_EAHsavgBMIz_item, X_EAHsavgBMIz_item).fit()

# Get summary of the regression model
print(mod_EAHsavgBMIz_item.summary())

assumption check for models

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsavgBMIz= mod_EAHsavgBMIz.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavgBMIz = durbin_watson(residuals_EAHsavgBMIz)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavgBMIz)
if 0 < dw_statistic_EAHsavgBMIz< 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavgBMIz.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavgBMIz.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavgBMIz.values, i) for i in range(X_EAHsavgBMIz.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavgBMIz.fittedvalues, y=residuals_EAHsavgBMIz)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()


bp_test = het_breuschpagan(residuals_EAHsavgBMIz,  mod_EAHsavgBMIz.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)") 

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavgBMIz, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavgBMIz, p_value_EAHsavgBMIz = shapiro(residuals_EAHsavgBMIz)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavgBMIz)
print("p-value:", p_value_EAHsavgBMIz)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsavgBMIz > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsavgBMIz_audio = mod_EAHsavgBMIz_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavgBMIz_audio = durbin_watson(residuals_EAHsavgBMIz_audio)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavgBMIz_audio)
if 0 < dw_statistic_EAHsavgBMIz_audio < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavgBMIz_audio.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavgBMIz_audio.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavgBMIz_audio.values, i) for i in range(X_EAHsavgBMIz_audio.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavgBMIz_audio.fittedvalues, y=residuals_EAHsavgBMIz_audio)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()


bp_test = het_breuschpagan(residuals_EAHsavgBMIz_audio,  mod_EAHsavgBMIz_audio.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)") 

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavgBMIz_audio, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavgBMIz_audio, p_value_EAHsavgBMIz_audio = shapiro(residuals_EAHsavgBMIz_audio)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavgBMIz_audio)
print("p-value:", p_value_EAHsavgBMIz_audio)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsavgBMIz_audio > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

In [ ]:
# checking for model assumptions
# Calculate the residuals
residuals_EAHsavgBMIz_item = mod_EAHsavgBMIz_audio.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavgBMIz_item = durbin_watson(residuals_EAHsavgBMIz_item)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavgBMIz_item)
if 0 < dw_statistic_EAHsavgBMIz_item < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavgBMIz_item.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavgBMIz_item.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavgBMIz_item.values, i) for i in range(X_EAHsavgBMIz_item.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavgBMIz_item.fittedvalues, y=residuals_EAHsavgBMIz_item)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()


bp_test = het_breuschpagan(residuals_EAHsavgBMIz_item,  mod_EAHsavgBMIz_item.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)") 

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavgBMIz_item, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavgBMIz_item, p_value_EAHsavgBMIz_item = shapiro(residuals_EAHsavgBMIz_item)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavgBMIz_item)
print("p-value:", p_value_EAHsavgBMIz_item)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsavgBMIz_item > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

## Partial Regression Plots: Baseline Intake (kcal) vs Follow-up adiposity (FMI)

In [ ]:
# Calculate the scaled standard error for intake values
sd_meal_g = df['v1_meal_g'].std()
sd_eah_g = df['v1_eah_g'].std()
sd_eah_sweet_g = df['v1_eah_sweet_g'].std()
sd_eah_sav_g = df['v1_eah_sav_g'].std()

sd_meal_kcal = df['v1_meal_kcal'].std()
sd_eah_kcal = df['v1_eah_kcal'].std()
sd_eah_sweet_kcal = df['v1_eah_sweet_kcal'].std()
sd_eah_sav_kcal = df['v1_eah_sav_kcal'].std()

sd_FMI = df['v7_FMI'].std()

In [ ]:
# define a function to compute the change in FMI for a 100 kcal increase
def change_in_FMI_for_100kcal_increase(coef, sd_FMI, sd_predictor):
    return coef * sd_FMI * (100/sd_predictor)

# Extract coefficients from the models
# Assuming each model has only one predictor, the coefficient will be for that predictor
coef_meal_fmi_kcal = mod_mealkcalFMI.params['v1_meal_kcal']
coef_eah_fmi_kcal = mod_EAHkcalFMI.params['v1_eah_kcal']
coef_eah_sweet_fmi_kcal = mod_EAHsweetkcalFMI.params['v1_eah_sweet_kcal']
coef_eah_sav_fmi_kcal = mod_EAHsavkcalFMI.params['v1_eah_sav_kcal']

# Calculate the change in FMI for a 100 kcal increase
change_meal_fmi_kcal = change_in_FMI_for_100kcal_increase(coef_meal_fmi_kcal, sd_FMI, sd_meal_kcal)
change_eah_fmi_kcal = change_in_FMI_for_100kcal_increase(coef_eah_fmi_kcal, sd_FMI, sd_eah_kcal)
change_eah_sweet_fmi_kcal = change_in_FMI_for_100kcal_increase(coef_eah_sweet_fmi_kcal, sd_FMI, sd_eah_sweet_kcal)
change_eah_sav_fmi_kcal= change_in_FMI_for_100kcal_increase(coef_eah_sav_fmi_kcal, sd_FMI, sd_eah_sav_kcal)

print("Change in FMI for 100 kcal increase in:")
print("Meal kcal:", change_meal_fmi_kcal)
print("EAH kcal:", change_eah_fmi_kcal)
print("EAH sweet kcal:", change_eah_sweet_fmi_kcal)
print("EAH sav kcal:", change_eah_sav_fmi_kcal)

In [ ]:
# Define a function to compute the scaled standard error for a 100 kcal increase
def scaled_se_for_100kcal_increase(se, sd_FMI, sd_predictor):
    return se * sd_FMI * (100 / sd_predictor)

# Extract standard errors from the models
# Assuming each model has an attribute 'bse' for standard errors
se_meal_fmi_kcal = mod_mealkcalFMI.bse['v1_meal_kcal']
se_eah_fmi_kcal = mod_EAHkcalFMI.bse['v1_eah_kcal']
se_eah_sweet_fmi_kcal = mod_EAHsweetkcalFMI.bse['v1_eah_sweet_kcal']
se_eah_sav_fmi_kcal = mod_EAHsavkcalFMI.bse['v1_eah_sav_kcal']

# Calculate the scaled standard error for a 100 kcal increase
scaled_se_meal_fmi_kcal = scaled_se_for_100kcal_increase(se_meal_fmi_kcal, sd_FMI, sd_meal_kcal)
scaled_se_eah_fmi_kcal = scaled_se_for_100kcal_increase(se_eah_fmi_kcal, sd_FMI, sd_eah_kcal)
scaled_se_eah_sweet_fmi_kcal = scaled_se_for_100kcal_increase(se_eah_sweet_fmi_kcal, sd_FMI, sd_eah_sweet_kcal)
scaled_se_eah_sav_fmi_kcal = scaled_se_for_100kcal_increase(se_eah_sav_fmi_kcal, sd_FMI, sd_eah_sav_kcal)

print("Scaled Standard Error for 100 kcal increase in:")
print("Meal kcal:", scaled_se_meal_fmi_kcal)
print("EAH kcal:", scaled_se_eah_fmi_kcal)
print("EAH sweet kcal:", scaled_se_eah_sweet_fmi_kcal)
print("EAH sav kcal:", scaled_se_eah_sav_fmi_kcal)

In [ ]:
# partial reg plots (intake kcal vs v7_FMI)
# Create a 2x2 subplot layout
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Partial regression meal (kcal) vs FMI
plot1 = sm.graphics.plot_partregress('v7_FMI', 'v1_meal_kcal', ['v1_FMI','sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'], data=df_linear_reg, obs_labels=False, ax=axes[0, 0])
axes[0, 0].set_xlabel("Baseline Std.Meal Intake(kcal)")
axes[0, 0].set_ylabel("Follow-up FMI: adiposity (kg/m2)")
axes[0, 0].set_title("A.Baseline Std Meal Intake vs. Follow-up Adiposity")
axes[0, 0].annotate(f"Partial Correlation: {mod_mealkcalFMI.params['v1_meal_kcal']:.3f}\nR-squared: {mod_mealkcalFMI.rsquared:.3f}\nP-value: {mod_mealkcalFMI.pvalues['v1_meal_kcal']:.3f}\nChange in FMI for 100 kcal increase: {change_meal_fmi_kcal:.3f} kg/m2", xy=(0.05, 0.8), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression EAH total (kcal) vs FMI
plot2 = sm.graphics.plot_partregress('v7_FMI', 'v1_eah_kcal', ['v1_FMI', 'sex','risk_status_mom', 'income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'], data=df_linear_reg, obs_labels=False, ax=axes[0, 1])
axes[0, 1].set_xlabel("Baseline EAH Intake (kcal)")
axes[0, 1].set_ylabel("Follow-up FMI: adiposity(kg/m2)")
axes[0, 1].set_title("B.Baseline Total EAH Intake vs. Follow-up Adiposity")
axes[0, 1].annotate(f"Partial Correlation: {mod_EAHkcalFMI.params['v1_eah_kcal']:.3f}\nR-squared: {mod_EAHkcalFMI.rsquared:.3f}\nP-value: {mod_EAHkcalFMI.pvalues['v1_eah_kcal']:.3f}\nChange in FMI for 100 kcal increase: {change_eah_fmi_kcal:.3f} kg/m2", xy=(0.05, 0.8), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression EAH sweet (kcal) vs FMI
sm.graphics.plot_partregress('v7_FMI', 'v1_eah_sweet_kcal', ['v1_FMI', 'sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'], data=df_linear_reg, obs_labels=False, ax=axes[1, 0])
axes[1, 0].set_xlabel("Baseline EAH Sweet Intake (kcal)")
axes[1, 0].set_ylabel("Follow-up FMI: adiposity(kg/m2)")
axes[1, 0].set_title("C.Baseline EAH Sweet Intake vs. Follow-up Adiposity")
axes[1, 0].annotate(f"Partial Correlation: {mod_EAHsweetkcalFMI.params['v1_eah_sweet_kcal']:.3f}\nR-squared: {mod_EAHsweetkcalFMI.rsquared:.3f}\nP-value: {mod_EAHsweetkcalFMI.pvalues['v1_eah_sweet_kcal']:.3f}\nChange in FMI for 100 kcal increase: {change_eah_sweet_fmi_kcal:.3f} kg/m2", xy=(0.05, 0.8), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression EAH savory (kcal) vs FMI
sm.graphics.plot_partregress('v7_FMI', 'v1_eah_sav_kcal', ['v1_FMI', 'sex','risk_status_mom', 'income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'], data=df_linear_reg, obs_labels=False, ax=axes[1, 1])
axes[1, 1].set_xlabel("Baseline EAH Savory Intake (kcal)")
axes[1, 1].set_ylabel("Follow-up FMI: adiposity(kg/m2)")
axes[1, 1].set_title("D.Baseline EAH Savory Intake vs. Follow-up Adiposity")
axes[1, 1].annotate(f"Partial Correlation: {mod_EAHsavkcalFMI.params['v1_eah_sav_kcal']:.3f}\nR-squared: {mod_EAHsavkcalFMI.rsquared:.3f}\nP-value: {mod_EAHsavkcalFMI.pvalues['v1_eah_sav_kcal']:.3f}\nChange in FMI for 100 kcal increase: {change_eah_sav_fmi_kcal:.3f} kg/m2", xy=(0.05, 0.8), xycoords='axes fraction', fontsize=12, color='blue')

fig.tight_layout()
plt.subplots_adjust(top=0.92)
plt.suptitle(
    "Partial Regression Plots: Intake Variables (kcal) as Predictors of Follow-Up Adiposity (FMI)", fontsize=16)
plt.show()

## Partial Regression Plots: Baseline Intake (kcal) vs Follow-up weight status (BMIz)

In [ ]:
# Extract the standard deviation only for predictor variables
sd_meal_kcal = df['v1_meal_kcal'].std()
sd_eah_kcal = df['v1_eah_kcal'].std()
sd_eah_sweet_kcal = df['v1_eah_sweet_kcal'].std()
sd_eah_sav_kcal = df['v1_eah_sav_kcal'].std()

# modify the function to compute the change in v7_bmi_z for a 100 kcal increase
def change_in_bmi_z_for_100kcal_increase(coef, sd_predictor):
    return coef * (100/sd_predictor)

# Extract coefficients from the models
coef_meal_bmi_kcal = mod_mealkcalBMIz.params['v1_meal_kcal']
coef_eah_bmi_kcal = mod_EAHkcalBMIz.params['v1_eah_kcal']
coef_eah_sweet_bmi_kcal = mod_EAHsweetkcalBMIz.params['v1_eah_sweet_kcal']
coef_eah_sav_bmi_kcal = mod_EAHsavkcalBMIz.params['v1_eah_sav_kcal']

# Calculate the change in v7_bmi_z for a 100 kcal increase
change_meal_bmi_kcal = change_in_bmi_z_for_100kcal_increase(coef_meal_bmi_kcal, sd_meal_kcal)
change_eah_bmi_kcal = change_in_bmi_z_for_100kcal_increase(coef_eah_bmi_kcal, sd_eah_kcal)
change_eah_sweet_bmi_kcal = change_in_bmi_z_for_100kcal_increase(coef_eah_sweet_bmi_kcal, sd_eah_sweet_kcal)
change_eah_sav_bmi_kcal = change_in_bmi_z_for_100kcal_increase(coef_eah_sav_bmi_kcal, sd_eah_sav_kcal)

print("Change in v7_bmi_z for 100 kcal increase in:")
print("Meal kcal:", change_meal_bmi_kcal)
print("EAH kcal:", change_eah_bmi_kcal)
print("EAH sweet kcal:", change_eah_sweet_bmi_kcal)
print("EAH sav kcal:", change_eah_sav_bmi_kcal)

In [ ]:
# Define a function to compute the scaled standard error for a 100 kcal increase
def scaled_se_for_100kcal_increase(se, sd_predictor):
    return se * (100 / sd_predictor)

# Extract standard errors from the models
se_meal_bmiz_kcal = mod_mealkcalBMIz.bse['v1_meal_kcal']
se_eah_bmiz_kcal = mod_EAHkcalBMIz.bse['v1_eah_kcal']
se_eah_sweet_bmiz_kcal = mod_EAHsweetkcalBMIz.bse['v1_eah_sweet_kcal']
se_eah_sav_bmiz_kcal = mod_EAHsavkcalBMIz.bse['v1_eah_sav_kcal']

# Calculate the scaled standard error for a 100 kcal increase
scaled_se_meal_bmiz_kcal = scaled_se_for_100kcal_increase(se_meal_bmiz_kcal, sd_meal_kcal)
scaled_se_eah_bmiz_kcal = scaled_se_for_100kcal_increase(se_eah_bmiz_kcal, sd_eah_kcal)
scaled_se_eah_sweet_bmiz_kcal = scaled_se_for_100kcal_increase(se_eah_sweet_bmiz_kcal, sd_eah_sweet_kcal)
scaled_se_eah_sav_bmiz_kcal = scaled_se_for_100kcal_increase(se_eah_sav_bmiz_kcal, sd_eah_sav_kcal)

print("Scaled Standard Error for 100 kcal increase in:")
print("Meal kcal:", scaled_se_meal_bmiz_kcal)
print("EAH kcal:", scaled_se_eah_bmiz_kcal)
print("EAH sweet kcal:", scaled_se_eah_sweet_bmiz_kcal)
print("EAH sav kcal:", scaled_se_eah_sav_bmiz_kcal)

In [ ]:
# partial reg plots (intake kcal vs v7_BMIz)
# Create a 2x2 subplot layout
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Partial regression meal (kcal) vs BMIz
plot1 = sm.graphics.plot_partregress('v7_bmi_z', 'v1_meal_kcal', ['bmi_z','sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'], data=df_linear_reg, obs_labels=False, ax=axes[0, 0])
axes[0, 0].set_xlabel("Baseline Std.Meal Intake (kcal)")
axes[0, 0].set_ylabel("Follow-up BMI-z")
axes[0, 0].set_title("A.Baseline Std Meal Intake vs. Follow-up BMI-z")
axes[0, 0].annotate(f"Partial Correlation: {mod_mealkcalBMIz.params['v1_meal_kcal']:.3f}\nR-squared: {mod_mealkcalBMIz.rsquared:.3f}\nP-value: {mod_mealkcalBMIz.pvalues['v1_meal_kcal']:.3f}\nChange in BMI-z for 100 kcal increase: {change_meal_bmi_kcal:.3f}", xy=(0.05, 0.8), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression EAH total (kcal) vs BMIz
plot2 = sm.graphics.plot_partregress('v7_bmi_z', 'v1_eah_kcal', ['bmi_z','sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'],  data=df_linear_reg, obs_labels=False, ax=axes[0, 1])
axes[0, 1].set_xlabel("Baseline EAH Intake (kcal)")
axes[0, 1].set_ylabel("Follow-up BMI-z")
axes[0, 1].set_title("B.Baseline Total EAH Intake vs. Follow-up BMI-z")
axes[0, 1].annotate(f"Partial Correlation: {mod_EAHkcalBMIz.params['v1_eah_kcal']:.3f}\nR-squared: {mod_EAHkcalBMIz.rsquared:.3f}\nP-value: {mod_EAHkcalBMIz.pvalues['v1_eah_kcal']:.3f}\nChange in BMI-z for 100 kcal increase: {change_eah_bmi_kcal:.3f}", xy=(0.05, 0.80), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression EAH sweet (kcal) vs BMIz
sm.graphics.plot_partregress('v7_bmi_z', 'v1_eah_sweet_kcal', ['bmi_z','sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'],  data=df_linear_reg, obs_labels=False, ax=axes[1, 0])
axes[1, 0].set_xlabel("Baseline EAH Sweet Intake (kcal)")
axes[1, 0].set_ylabel("Follow-up BMI-z")
axes[1, 0].set_title("C.Baseline EAH Sweet Intake vs. Follow-up BMI-z")
axes[1, 0].annotate(f"Partial Correlation: {mod_EAHsweetkcalBMIz.params['v1_eah_sweet_kcal']:.3f}\nR-squared: {mod_EAHsweetkcalBMIz.rsquared:.3f}\nP-value: {mod_EAHsweetkcalBMIz.pvalues['v1_eah_sweet_kcal']:.3f}\nChange in BMI-z for 100 kcal increase: {change_eah_sweet_bmi_kcal:.3f}", xy=(0.05, 0.80), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression EAH savory (kcal) vs FMI
sm.graphics.plot_partregress('v7_bmi_z', 'v1_eah_sav_kcal', ['bmi_z','sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'],  data=df_linear_reg, obs_labels=False, ax=axes[1, 1])
axes[1, 1].set_xlabel("Baseline EAH Savory Intake (kcal)")
axes[1, 1].set_ylabel("Follow-up BMI-z")
axes[1, 1].set_title("D.Baseline EAH Savory Intake vs. Follow-up BMI-z")
axes[1, 1].annotate(f"Partial Correlation: {mod_EAHsavkcalBMIz.params['v1_eah_sav_kcal']:.3f}\nR-squared: {mod_EAHsavkcalBMIz.rsquared:.3f}\nP-value: {mod_EAHsavkcalBMIz.pvalues['v1_eah_sav_kcal']:.3f}\nChange in BMI-z for 100 kcal increase: {change_eah_sav_bmi_kcal:.3f}", xy=(0.05, 0.80), xycoords='axes fraction', fontsize=12, color='blue')

fig.tight_layout()
plt.subplots_adjust(top=0.92)
plt.suptitle("Partial Regression Plots: Intake Variable (kcal) as Predictors of Follow-up BMI-z score", fontsize=16)
plt.show()


## Partial Regression Plots: Baseline Intake (gram) vs Follow-up adiposity (FMI)

In [ ]:
# Extract standard deviations from original dataframe
sd_FMI = df['v7_FMI'].std()
sd_meal_g = df['v1_meal_g'].std()
sd_eah_g = df['v1_eah_g'].std()
sd_eah_sweet_g = df['v1_eah_sweet_g'].std()
sd_eah_sav_g = df['v1_eah_sav_g'].std()

# Function to compute the change in FMI for a 100g increase
def change_in_FMI_for_100g_increase(coef, sd_FMI, sd_predictor):
    return coef * sd_FMI * (100/sd_predictor)

# Extract coefficients from the models
coef_meal_g_fmi = mod_mealgFMI.params['v1_meal_g']  # Assuming the predictors in your models are now the gram columns
coef_eah_g_fmi = mod_EAHgFMI.params['v1_eah_g']
coef_eah_sweet_g_fmi = mod_EAHsweetgFMI.params['v1_eah_sweet_g']
coef_eah_sav_g_fmi = mod_EAHsavgFMI.params['v1_eah_sav_g']

# Calculate the change in FMI for a 100g increase
change_meal_g_fmi = change_in_FMI_for_100g_increase(coef_meal_g_fmi, sd_FMI, sd_meal_g)
change_eah_g_fmi = change_in_FMI_for_100g_increase(coef_eah_g_fmi, sd_FMI, sd_eah_g)
change_eah_sweet_g_fmi = change_in_FMI_for_100g_increase(coef_eah_sweet_g_fmi, sd_FMI, sd_eah_sweet_g)
change_eah_sav_g_fmi = change_in_FMI_for_100g_increase(coef_eah_sav_g_fmi, sd_FMI, sd_eah_sav_g)

print("Change in FMI for 100g increase in:")
print("Meal g:", change_meal_g_fmi)
print("EAH g:", change_eah_g_fmi)
print("EAH sweet g:", change_eah_sweet_g_fmi)
print("EAH sav g:", change_eah_sav_g_fmi)


In [ ]:
# partial reg plots (intake gram vs v7_FMI)
# Create a 2x2 subplot layout
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Partial regression meal (g) vs BMIz
plot1 = sm.graphics.plot_partregress('v7_FMI', 'v1_meal_g', ['v1_FMI','sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'], data=df_linear_reg, obs_labels=False, ax=axes[0, 0])
axes[0, 0].set_xlabel("Baseline Std.Meal Intake (gram)")
axes[0, 0].set_ylabel("Follow-up Adiposity: FMI (kg/m2)")
axes[0, 0].set_title("A.Baseline Std Meal Intake vs. Follow-up Adiposity")
axes[0, 0].annotate(f"Partial Correlation: {mod_mealgFMI.params['v1_meal_g']:.3f}\nR-squared: {mod_mealgFMI.rsquared:.3f}\nP-value: {mod_mealgFMI.pvalues['v1_meal_g']:.3f}\nChange in FMI for 100g increase: {change_meal_g_fmi:.3f}kg/m2", xy=(0.05, 0.80), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression EAH total (gram) vs FMI
plot2 = sm.graphics.plot_partregress('v7_FMI', 'v1_eah_g', ['v1_FMI','sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'], data=df_linear_reg, obs_labels=False, ax=axes[0, 1])
axes[0, 1].set_xlabel("Baseline EAH Intake (grams)")
axes[0, 1].set_ylabel("Follow-up Adiposity:FMI (kg/m2)")
axes[0, 1].set_title("B.Baseline Total EAH Intake vs. Follow-up Adiposity")
axes[0, 1].annotate(f"Partial Correlation: {mod_EAHgFMI.params['v1_eah_g']:.3f}\nR-squared: {mod_EAHgFMI.rsquared:.3f}\nP-value: {mod_EAHgFMI.pvalues['v1_eah_g']:.3f}\nChange in FMI for 100g increase: {change_eah_g_fmi:.3f}kg/m2", xy=(
    0.05, 0.80), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression EAH sweet (gram) vs FMI
sm.graphics.plot_partregress('v7_FMI', 'v1_eah_sweet_g', ['v1_FMI','sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'], data=df_linear_reg, obs_labels=False, ax=axes[1, 0])
axes[1, 0].set_xlabel("Baseline EAH Sweet Intake (grams)")
axes[1, 0].set_ylabel("Follow-up Adiposity:FMI (kg/m2)")
axes[1, 0].set_title("C.Baseline EAH Sweet Intake vs. Follow-up Adiposity")
axes[1, 0].annotate(f"Partial Correlation: {mod_EAHsweetgFMI.params['v1_eah_sweet_g']:.3f}\nR-squared: {mod_EAHsweetgFMI.rsquared:.3f}\nP-value: {mod_EAHsweetgFMI.pvalues['v1_eah_sweet_g']:.3f}\nChange in FMI for 100g increase: {change_eah_sweet_g_fmi:.3f}kg/m2", xy=(
    0.05, 0.80), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression EAH savory (gram) vs FMI
sm.graphics.plot_partregress('v7_FMI', 'v1_eah_sav_g', ['v1_FMI','sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'], data=df_linear_reg, obs_labels=False, ax=axes[1, 1])
axes[1, 1].set_xlabel("Baseline EAH Sweet Intake (grams)")
axes[1, 1].set_ylabel("Follow-up Adiposity:FMI (kg/m2)")
axes[1, 1].set_title("D.Baseline EAH Savory Intake vs. Follow-up Adiposity")
axes[1, 1].annotate(f"Partial Correlation: {mod_EAHsavgFMI.params['v1_eah_sav_g']:.3f}\nR-squared: {mod_EAHsavgFMI.rsquared:.3f}\nP-value: {mod_EAHsavgFMI.pvalues['v1_eah_sav_g']:.3f}\nChange in FMI for 100g increase: {change_eah_sav_g_fmi:.3f}kg/m2", xy=(
    0.05, 0.80), xycoords='axes fraction', fontsize=12, color='blue')

fig.tight_layout()
plt.suptitle(
    "Partial Correlation Plots: Intake Variables (grams) as Predictors of Follow-Up Adiposity (FMI)", fontsize=16)
plt.subplots_adjust(top=0.92)

## Partial Regression Plots: Baseline Intake (gram) vs Follow-up weight status (BMIz)

In [ ]:
# Extract the standard deviation only for predictor variables
sd_meal_g = df['v1_meal_g'].std()
sd_eah_g= df['v1_eah_g'].std()
sd_eah_sweet_g = df['v1_eah_sweet_g'].std()
sd_eah_sav_g = df['v1_eah_sav_g'].std()

# modify the function to compute the change in v7_bmi_z for a 100 kcal increase
def change_in_bmi_z_for_100g_increase(coef, sd_predictor):
    return coef * (100/sd_predictor)

# Extract coefficients from the models
coef_meal_bmi_g = mod_mealgBMIz.params['v1_meal_g']
coef_eah_bmi_g = mod_EAHgBMIz.params['v1_eah_g']
coef_eah_sweet_bmi_g = mod_EAHsweetgBMIz.params['v1_eah_sweet_g']
coef_eah_sav_bmi_g = mod_EAHsavgBMIz.params['v1_eah_sav_g']

# Calculate the change in v7_bmi_z for a 100 kcal increase
change_meal_bmi_g = change_in_bmi_z_for_100kcal_increase(coef_meal_bmi_g, sd_meal_g)
change_eah_bmi_g = change_in_bmi_z_for_100kcal_increase(coef_eah_bmi_g, sd_eah_g)
change_eah_sweet_bmi_g = change_in_bmi_z_for_100kcal_increase(coef_eah_sweet_bmi_g, sd_eah_sweet_g)
change_eah_sav_bmi_g = change_in_bmi_z_for_100kcal_increase(coef_eah_sav_bmi_g, sd_eah_sav_g)

print("Change in v7_bmi_z for 100 kcal increase in:")
print("Meal kcal:", change_meal_bmi_g)
print("EAH kcal:", change_eah_bmi_g)
print("EAH sweet kcal:", change_eah_sweet_bmi_g)
print("EAH sav kcal:", change_eah_sav_bmi_g)

In [ ]:
# partial reg plots (intake gram vs v7_bmi_z)
# Create a 2x2 subplot layout
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Partial regression meal (gram) vs FMI
plot1 = sm.graphics.plot_partregress('v7_bmi_z', 'v1_meal_g', ['bmi_z','sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'], data=df_linear_reg, obs_labels=False, ax=axes[0, 0])
axes[0, 0].set_xlabel("Baseline Std.Meal Intake (grams)")
axes[0, 0].set_ylabel("Follow-up BMI-z score")
axes[0, 0].set_title("A.Baseline Std Meal Intake vs. Follow-up BMI-z")
axes[0, 0].annotate(f"Partial Correlation: {mod_mealgBMIz.params['v1_meal_g']:.3f}\nR-squared: {mod_mealgBMIz.rsquared:.3f}\nP-value: {mod_mealgBMIz.pvalues['v1_meal_g']:.3f}\nChange in BMI-z for 100g increase: {change_meal_bmi_g:.3f}", xy=(
    0.05, 0.8), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression EAH total (gram) vs FMI
plot2 = sm.graphics.plot_partregress('v7_bmi_z', 'v1_eah_g', ['bmi_z','sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'], data=df_linear_reg, obs_labels=False, ax=axes[0, 1])
axes[0, 1].set_xlabel("Baseline EAH Intake (grams)")
axes[0, 1].set_ylabel("Follow-up BMI-z score")
axes[0, 1].set_title("B.Baseline Total EAH Intake vs. Follow-up BMI-z")
axes[0, 1].annotate(f"Partial Correlation: {mod_EAHgBMIz.params['v1_eah_g']:.3f}\nR-squared: {mod_EAHgBMIz.rsquared:.3f}\nP-value: {mod_EAHgBMIz.pvalues['v1_eah_g']:.3f}\nChange in BMI-z for 100g increase: {change_eah_bmi_g:.3f}", xy=(
    0.05, 0.8), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression EAH sweet (gram) vs FMI
sm.graphics.plot_partregress('v7_bmi_z', 'v1_eah_sweet_g', ['bmi_z','sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'], data=df_linear_reg, obs_labels=False, ax=axes[1, 0])
axes[1, 0].set_xlabel("Baseline EAH Sweet Intake (grams)")
axes[1, 0].set_ylabel("Follow-up BMI-z score")
axes[1, 0].set_title("C.Baseline EAH Sweet Intake vs. Follow-up BMI-z")
axes[1, 0].annotate(f"Partial Correlation: {mod_EAHsweetgBMIz.params['v1_eah_sweet_g']:.3f}\nR-squared: {mod_EAHsweetgBMIz.rsquared:.3f}\nP-value: {mod_EAHsweetgBMIz.pvalues['v1_eah_sweet_g']:.3f}\nChange in BMI-z for 100g increase: {change_eah_sweet_bmi_g:.3f}", xy=(
    0.05, 0.8), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression EAH savory (gram) vs FMI
sm.graphics.plot_partregress('v7_bmi_z', 'v1_eah_sav_g', ['bmi_z','sex','risk_status_mom','income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff'], data=df_linear_reg, obs_labels=False, ax=axes[1, 1])
axes[1, 1].set_xlabel("Baseline EAH Sweet Intake (grams)")
axes[1, 1].set_ylabel("Follow-up BMI-z score")
axes[1, 1].set_title("D.Baseline EAH Savory Intake vs. Follow-up BMI-z")
axes[1, 1].annotate(f"Partial Correlation: {mod_EAHsavgBMIz.params['v1_eah_sav_g']:.3f}\nR-squared: {mod_EAHsavgBMIz.rsquared:.3f}\nP-value: {mod_EAHsavgBMIz.pvalues['v1_eah_sav_g']:.3f}\nChange in BMI-z for 100g increase: {change_eah_sav_bmi_g:.3f}", xy=(
    0.05, 0.8), xycoords='axes fraction', fontsize=12, color='blue')

fig.tight_layout()
plt.suptitle("Partial Correlation Plots: Intake Variables (grams) as Predictors of Follow-Up Body Weight (BMI-z score)", fontsize=16)
plt.subplots_adjust(top=0.92)
plt.show()


# SE reverse-scaling

In [ ]:
scaled_se_meal_fmi_kcal = mod_mealkcalFMI.bse['v1_meal_kcal'] * sd_FMI * (100 / sd_meal_kcal)
scaled_se_eah_fmi_kcal = mod_EAHkcalFMI.bse['v1_eah_kcal'] * sd_FMI * (100 / sd_eah_kcal)
scaled_se_eah_sweet_fmi_kcal = mod_EAHsweetkcalFMI.bse['v1_eah_sweet_kcal'] * sd_FMI * (100 / sd_eah_sweet_kcal)
scaled_se_eah_sav_fmi_kcal = mod_EAHsavkcalFMI.bse['v1_eah_sav_kcal'] * sd_FMI * (100 / sd_eah_sav_kcal)


scaled_se_meal_fmi_g = mod_mealgFMI.bse['v1_meal_g'] * sd_FMI * (100 / sd_meal_g)
scaled_se_eah_fmi_g = mod_EAHgFMI.bse['v1_eah_g'] * sd_FMI * (100 / sd_eah_g)
scaled_se_eah_sweet_fmi_g = mod_EAHsweetgFMI.bse['v1_eah_sweet_g'] * sd_FMI * (100 / sd_eah_sweet_g)
scaled_se_eah_sav_fmi_g = mod_EAHsavgFMI.bse['v1_eah_sav_g'] * sd_FMI * (100 / sd_eah_sav_g)


scaled_se_meal_bmiz_kcal = mod_mealkcalBMIz.bse['v1_meal_kcal'] * (100 / sd_meal_kcal)
scaled_se_eah_bmiz_kcal = mod_EAHkcalBMIz.bse['v1_eah_kcal'] * (100 / sd_eah_kcal)
scaled_se_eah_sweet_bmiz_kcal = mod_EAHsweetkcalBMIz.bse['v1_eah_sweet_kcal'] * (100 / sd_eah_sweet_kcal)
scaled_se_eah_sav_bmiz_kcal = mod_EAHsavkcalBMIz.bse['v1_eah_sav_kcal'] * (100 / sd_eah_sav_kcal)

scaled_se_meal_bmiz_g = mod_mealgBMIz.bse['v1_meal_g'] * (100 / sd_meal_g)
scaled_se_eah_bmiz_g = mod_EAHgBMIz.bse['v1_eah_g'] * (100 / sd_eah_g)
scaled_se_eah_sweet_bmiz_g = mod_EAHsweetgBMIz.bse['v1_eah_sweet_g'] * (100 / sd_eah_sweet_g)
scaled_se_eah_sav_bmiz_g = mod_EAHsavgBMIz.bse['v1_eah_sav_g'] * (100 / sd_eah_sav_g)


print("Change in SE of FMI for 100 kcal increase in:")
print("SE of beta Meal kcal:{:.2f}".format(scaled_se_meal_fmi_kcal))
print("SE of beta EAH kcal:{:.2f}".format(scaled_se_eah_fmi_kcal))
print("SE of beta EAH sweet kcal:{:.2f}".format(scaled_se_eah_sweet_fmi_kcal))
print("SE of beta EAH sav kcal:{:.2f}".format(scaled_se_eah_sav_fmi_kcal))


print("Change in SE of FMI for 100 g increase in:")
print("SE of beta Meal g:{:.2f}".format(scaled_se_meal_fmi_g))
print("SE of beta EAH g:{:.2f}".format(scaled_se_eah_fmi_g))
print("SE of beta EAH sweet g:{:.2f}".format(scaled_se_eah_sweet_fmi_g))
print("SE of beta EAH sav g:{:.2f}".format(scaled_se_eah_sav_fmi_g))

print("Change in SE of BMIz for 100 kcal increase in:")
print("SE of beta Meal kcal:{:.2f}".format(scaled_se_meal_bmiz_kcal))
print("SE of beta EAH kcal:{:.2f}".format(scaled_se_eah_bmiz_kcal))
print("SE of beta EAH sweet kcal:{:.2f}".format(scaled_se_eah_sweet_bmiz_kcal))
print("SE of beta EAH sav kcal:{:.2f}".format(scaled_se_eah_sav_bmiz_kcal))

print("Change in SE of BMIz for 100 g increase in:")
print("SE of beta Meal g:{:.2f}".format(scaled_se_meal_bmiz_g))
print("SE of beta EAH g:{:.2f}".format(scaled_se_eah_bmiz_g))
print("SE of beta EAH sweet g:{:.2f}".format(scaled_se_eah_sweet_bmiz_g))
print("SE of beta EAH sav g:{:.2f}".format(scaled_se_eah_sav_bmiz_g))

## Scatterplots for baseline vs follow-up intake measures(kcal)

In [ ]:
# Create a 2x2 grid of subplots
fig, axs = plt.subplots(2, 2, figsize=(12, 10))

# Scatter Plot for Meal Intake
axs[0, 0].scatter(df['v1_meal_kcal'], df['v7_meal_kcal'], s=50)
axs[0, 0].set_xlabel('Meal kcal Intake at Baseline')
axs[0, 0].set_ylabel('Meal kcal Intake at Follow-up')
axs[0, 0].set_title('A.Meal kcal Intake')

# Fit a linear regression model for Meal Intake
x_meal = df['v1_meal_kcal']
y_meal = df['v7_meal_kcal']
coeff_meal = np.polyfit(x_meal, y_meal, 1)
polynomial_meal = np.poly1d(coeff_meal)
correlation_matrix_meal = np.corrcoef(x_meal, y_meal)
correlation_xy_meal = correlation_matrix_meal[0, 1]
r_squared_meal = correlation_xy_meal ** 2
textstr_meal = f'R-squared = {r_squared_meal:.2f}\nCoefficient = {coeff_meal[0]:.4f}'
axs[0, 0].text(0.05, 0.95, textstr_meal, transform=axs[0, 0].transAxes, fontsize=12,
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
axs[0, 0].plot(x_meal, polynomial_meal(x_meal), color='black')

# Scatter Plot for EAH Intake
axs[0, 1].scatter(df['v1_eah_kcal'], df['v7_eah_kcal'], s=50)
axs[0, 1].set_xlabel('EAH kcal Intake at Baseline')
axs[0, 1].set_ylabel('EAH kcal Intake at Follow-up')
axs[0, 1].set_title('B.EAH kcal Intake')

# Fit a linear regression model for EAH Intake
x_eah = df['v1_eah_kcal']
y_eah = df['v7_eah_kcal']
coeff_eah = np.polyfit(x_eah, y_eah, 1)
polynomial_eah = np.poly1d(coeff_eah)
correlation_matrix_eah = np.corrcoef(x_eah, y_eah)
correlation_xy_eah = correlation_matrix_eah[0, 1]
r_squared_eah = correlation_xy_eah ** 2
textstr_eah = f'R-squared = {r_squared_eah:.2f}\nCoefficient = {coeff_eah[0]:.4f}'
axs[0, 1].text(0.05, 0.95, textstr_eah, transform=axs[0, 1].transAxes, fontsize=12,
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
axs[0, 1].plot(x_eah, polynomial_eah(x_eah), color='black')


# Scatter Plot for Sweet EAH Intake
axs[1, 0].scatter(df['v1_eah_sweet_kcal'], df['v7_eah_sweet_kcal'], s=50)
axs[1, 0].set_xlabel('Sweet EAH kcal Intake at Baseline')
axs[1, 0].set_ylabel('Sweet EAH kcal Intake at Follow-up')
axs[1, 0].set_title('C.Sweet EAH kcal Intake')

# Fit a linear regression model for Sweet EAH Intake
x_sweet = df['v1_eah_sweet_kcal']
y_sweet = df['v7_eah_sweet_kcal']
coeff_sweet = np.polyfit(x_sweet, y_sweet, 1)
polynomial_sweet = np.poly1d(coeff_sweet)
correlation_matrix_sweet = np.corrcoef(x_sweet, y_sweet)
correlation_xy_sweet = correlation_matrix_sweet[0, 1]
r_squared_sweet = correlation_xy_sweet ** 2
textstr_sweet = f'R-squared = {r_squared_sweet:.2f}\nCoefficient = {coeff_sweet[0]:.4f}'
axs[1, 0].text(0.05, 0.95, textstr_sweet, transform=axs[1, 0].transAxes, fontsize=12,
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
axs[1, 0].plot(x_sweet, polynomial_sweet(x_sweet), color='black')

# Scatter Plot for Savory EAH Intake
axs[1, 1].scatter(df['v1_eah_sav_kcal'], df['v7_eah_sav_kcal'], s=50)
axs[1, 1].set_xlabel('Savory EAH kcal Intake at Baseline')
axs[1, 1].set_ylabel('Savory EAH kcal Intake at Follow-up')
axs[1, 1].set_title('D.Savory EAH kcal Intake')

# Fit a linear regression model for Savory EAH Intake
x_savory = df['v1_eah_sav_kcal']
y_savory = df['v7_eah_sav_kcal']
coeff_savory = np.polyfit(x_savory, y_savory, 1)
polynomial_savory = np.poly1d(coeff_savory)
correlation_matrix_savory = np.corrcoef(x_savory, y_savory)
correlation_xy_savory = correlation_matrix_savory[0, 1]
r_squared_savory = correlation_xy_savory ** 2
textstr_savory = f'R-squared = {r_squared_savory:.2f}\nCoefficient = {coeff_savory[0]:.4f}'
axs[1, 1].text(0.05, 0.95, textstr_savory, transform=axs[1, 1].transAxes, fontsize=12,
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
axs[1, 1].plot(x_savory, polynomial_savory(x_savory), color='black')

plt.tight_layout()
plt.suptitle(
    "Association of Intake between Baseline and Follow-up (kcal)", fontsize=16)
plt.subplots_adjust(top=0.92)
plt.show()

## Scatterplots for baseline vs follow-up intake measures(grams)

In [ ]:
# Create a 2x2 grid of subplots
fig, axs = plt.subplots(2, 2, figsize=(12, 10))

# Scatter Plot for Meal Intake
axs[0, 0].scatter(df['v1_meal_g'], df['v7_meal_g'], s=50)
axs[0, 0].set_xlabel('Meal gram Intake at Baseline')
axs[0, 0].set_ylabel('Meal gram Intake at Follow-up')
axs[0, 0].set_title('A.Meal gram intake')

# Fit a linear regression model for Meal Intake
x_meal = df['v1_meal_g']
y_meal = df['v7_meal_g']
coeff_meal = np.polyfit(x_meal, y_meal, 1)
polynomial_meal = np.poly1d(coeff_meal)
correlation_matrix_meal = np.corrcoef(x_meal, y_meal)
correlation_xy_meal = correlation_matrix_meal[0, 1]
r_squared_meal = correlation_xy_meal ** 2
textstr_meal = f'R-squared = {r_squared_meal:.2f}\nCoefficient = {coeff_meal[0]:.4f}'
axs[0, 0].text(0.05, 0.95, textstr_meal, transform=axs[0, 0].transAxes, fontsize=12,
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
axs[0, 0].plot(x_meal, polynomial_meal(x_meal), color='black')# Scatter Plot for EAH Intake
axs[0, 1].scatter(df['v1_eah_g'], df['v7_eah_g'], s=50)
axs[0, 1].set_xlabel('EAH gram Intake at Baseline')
axs[0, 1].set_ylabel('EAH gram Intake at Follow-up')
axs[0, 1].set_title('B.EAH gram intake')

# Fit a linear regression model for EAH Intake
x_eah = df['v1_eah_g']
y_eah = df['v7_eah_g']
coeff_eah = np.polyfit(x_eah, y_eah, 1)
polynomial_eah = np.poly1d(coeff_eah)
correlation_matrix_eah = np.corrcoef(x_eah, y_eah)
correlation_xy_eah = correlation_matrix_eah[0, 1]
r_squared_eah = correlation_xy_eah ** 2
textstr_eah = f'R-squared = {r_squared_eah:.2f}\nCoefficient = {coeff_eah[0]:.4f}'
axs[0, 1].text(0.05, 0.95, textstr_eah, transform=axs[0, 1].transAxes, fontsize=12,
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
axs[0, 1].plot(x_eah, polynomial_eah(x_eah), color='black')

# Scatter Plot for Sweet EAH Intake
axs[1, 0].scatter(df['v1_eah_sweet_g'], df['v7_eah_sweet_g'], s=50)
axs[1, 0].set_xlabel('Sweet EAH gram Intake at Baseline')
axs[1, 0].set_ylabel('Sweet EAH gram Intake at Follow-up')
axs[1, 0].set_title('C.Sweet EAH gram intake')

# Fit a linear regression model for Sweet EAH Intake
x_sweet = df['v1_eah_sweet_g']
y_sweet = df['v7_eah_sweet_g']
coeff_sweet = np.polyfit(x_sweet, y_sweet, 1)
polynomial_sweet = np.poly1d(coeff_sweet)
correlation_matrix_sweet = np.corrcoef(x_sweet, y_sweet)
correlation_xy_sweet = correlation_matrix_sweet[0, 1]
r_squared_sweet = correlation_xy_sweet ** 2
textstr_sweet = f'R-squared = {r_squared_sweet:.2f}\nCoefficient = {coeff_sweet[0]:.4f}'
axs[1, 0].text(0.05, 0.95, textstr_sweet, transform=axs[1, 0].transAxes, fontsize=12,
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
axs[1, 0].plot(x_sweet, polynomial_sweet(x_sweet), color='black')

# Scatter Plot for Savory EAH Intake
axs[1, 1].scatter(df['v1_eah_sav_g'], df['v7_eah_sav_g'], s=50)
axs[1, 1].set_xlabel('Savory EAH gram Intake at Baseline')
axs[1, 1].set_ylabel('Savory EAH gram Intake at Follow-up')
axs[1, 1].set_title('D.Savory EAH gram intake')

# Fit a linear regression model for Savory EAH Intake
x_savory = df['v1_eah_sav_g']
y_savory = df['v7_eah_sav_g']
coeff_savory = np.polyfit(x_savory, y_savory, 1)
polynomial_savory = np.poly1d(coeff_savory)
correlation_matrix_savory = np.corrcoef(x_savory, y_savory)
correlation_xy_savory = correlation_matrix_savory[0, 1]
r_squared_savory = correlation_xy_savory ** 2
textstr_savory = f'R-squared = {r_squared_savory:.2f}\nCoefficient = {coeff_savory[0]:.4f}'
axs[1, 1].text(0.05, 0.95, textstr_savory, transform=axs[1, 1].transAxes, fontsize=12,
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))
axs[1, 1].plot(x_savory, polynomial_savory(x_savory), color='black')

plt.tight_layout()
plt.suptitle("Association of Intake between Baseline and Follow-up (gram)", fontsize=16)
plt.subplots_adjust(top=0.92)
plt.show()

## DEMOGRAPHIC TABLE - separate for continuous and categorical

categorical vars

In [ ]:
# Assuming 'df' is your original DataFrame with data

categorical_vars = ['parent_respondent', 'parent_ed', 'income', 'risk_status_mom',
                    'sex', 'ethnicity', 'race', 'pds_tanner_cat']

# Initialize an empty list to store the results
categorical_data = []

# Loop through each categorical column and calculate percentages and counts
for column in categorical_vars:
    counts = df[column].value_counts()
    percentages = counts / counts.sum() * 100
    for category, count in counts.items():
        categorical_data.append({
            'Variable': column,
            'Category': category,
            'Percentage': percentages[category],
            'Count': count
        })

# Convert the list of dictionaries into a DataFrame
categorical_table = pd.DataFrame(categorical_data)

print(categorical_table)

# Save to Excel
categorical_table.to_excel('cat_table.xlsx', index=False)

continuous vars

In [ ]:
# Create a new DataFrame for the continuous table
continuous_vars = ['age_yr', 'bmi_percentile', 'bmi_z', 'v7_age_yr', 'v7_bmi_percentile', 'v7_bmi_z', 'age_diff']
continuous_table = pd.DataFrame()

# Continuous variables - mean +/- SD and output range
for col in continuous_vars:
    continuous_table[f'{col}_mean'] = [df[col].mean()]
    continuous_table[f'{col}_SD'] = [df[col].std()]
    continuous_table[f'{col}_min'] = [df[col].min()]
    continuous_table[f'{col}_max'] = [df[col].max()]

# Display the continuous table
print("\nContinuous Table:")
print(continuous_table)
continuous_table.to_excel('cont_table.xlsx')

# Intake vars analysis, Mean,SD, Max-Min

In [ ]:
intake_vars = ['v1_eah_kcal', 'v7_eah_kcal', 'v1_meal_kcal', 'v7_meal_kcal',
               'v1_eah_g', 'v7_eah_g', 'v1_meal_g', 'v7_meal_g',
               'v1_eah_sweet_kcal', 'v7_eah_sweet_kcal', 'v1_eah_sav_kcal', 'v7_eah_sav_kcal',
               'v1_eah_sweet_g', 'v7_eah_sweet_g', 'v1_eah_sav_g', 'v7_eah_sav_g']

# Initialize an empty DataFrame to store the Intake Table
intake_table = pd.DataFrame(
    columns=['Intake Variable', 'Mean', 'Min', 'Max', 'SD'])

# Function to calculate statistics and append to the intake_table


def add_to_intake_table(column_name, data_column):
    mean_val = data_column.mean()
    min_val = data_column.min()
    max_val = data_column.max()
    sd_val = data_column.std()

    intake_table.loc[len(intake_table)] = [column_name, mean_val, min_val, max_val, sd_val]


# Calculate statistics for intake variables and add to intake_table
for column in intake_vars:
    add_to_intake_table(column, df[column])

print(intake_table)
intake_table.to_excel('intake_table.xlsx', index=False)

# Intake variables independent T test between low vs risk status

In [ ]:
from scipy.stats import ttest_ind

# Filter data for each risk group
low_risk_data = df[df["risk_status_mom"] == 0]
high_risk_data = df[df["risk_status_mom"] == 1]

# List of continuous variables to test
continuous_variables = ["v1_eah_g", "v1_eah_kcal", "v7_eah_g", "v7_eah_kcal", "v1_meal_g", "v1_meal_kcal", "v7_meal_g", "v7_meal_kcal","v1_eah_sweet_g", "v1_eah_sweet_kcal", "v7_eah_sweet_g", "v7_eah_sweet_kcal","v1_eah_sav_g", "v1_eah_sav_kcal", "v7_eah_sav_g", "v7_eah_sav_kcal"]

# Create a DataFrame to store results
results = []

# Loop through each continuous variable
for variable in continuous_variables:
    t_stat, p_value = ttest_ind(low_risk_data[variable], high_risk_data[variable])
    
    result_entry = {
        "Variable": variable,
        "T-statistic": t_stat,
        "P-value": p_value
    }
    
    results.append(result_entry)

# Convert results list to DataFrame
results_df = pd.DataFrame(results)

# Export results to Excel
results_df.to_excel("t_test_eahvsmeal.xlsx", index=False)

## Analysis by risk status

In [ ]:
# Filter data for each risk group
low_risk_data = df[df["risk_status_mom"] == 0]
high_risk_data = df[df["risk_status_mom"] == 1]

# List of continuous variables to calculate statistics for
continuous_variables = [
    "v1_eah_g", "v1_eah_kcal", "v7_eah_g", "v7_eah_kcal",
    "v1_meal_g", "v1_meal_kcal", "v7_meal_g", "v7_meal_kcal",
    "v1_eah_sweet_g", "v1_eah_sweet_kcal", "v7_eah_sweet_g", "v7_eah_sweet_kcal",
    "v1_eah_sav_g", "v1_eah_sav_kcal", "v7_eah_sav_g", "v7_eah_sav_kcal"
]

# Create a DataFrame to store statistics
statistics = []

# Calculate statistics (mean, SD, min, max) for each variable based on risk status
for variable in continuous_variables:
    low_risk_mean = low_risk_data[variable].mean()
    low_risk_sd = low_risk_data[variable].std()
    low_risk_min = low_risk_data[variable].min()
    low_risk_max = low_risk_data[variable].max()

    high_risk_mean = high_risk_data[variable].mean()
    high_risk_sd = high_risk_data[variable].std()
    high_risk_min = high_risk_data[variable].min()
    high_risk_max = high_risk_data[variable].max()

    result_entry = {
        "Variable": variable,
        "Low Risk Mean": low_risk_mean,
        "Low Risk SD": low_risk_sd,
        "Low Risk Min": low_risk_min,
        "Low Risk Max": low_risk_max,
        "High Risk Mean": high_risk_mean,
        "High Risk SD": high_risk_sd,
        "High Risk Min": high_risk_min,
        "High Risk Max": high_risk_max
    }

    statistics.append(result_entry)

# Convert statistics list to DataFrame
statistics_df = pd.DataFrame(statistics)

# Export statistics to Excel
statistics_df.to_excel("statistics_by_risk_status.xlsx", index=False)

# T-test for intake vars at T1 and T2

In [ ]:
import pandas as pd
from scipy.stats import ttest_rel

# Assuming your dataframe is named df
variables_to_compare = [
    ('meal_kcal', 'v1_meal_kcal', 'v7_meal_kcal'),
    ('eah_kcal', 'v1_eah_kcal', 'v7_eah_kcal'),
    ('sweet_kcal', 'v1_eah_sweet_kcal', 'v7_eah_sweet_kcal'),
    ('sav_kcal', 'v1_eah_sav_kcal', 'v7_eah_sav_kcal'),
    ('sav_g', 'v1_eah_sav_g', 'v7_eah_sav_g'),
    ('sweet_g', 'v1_eah_sweet_g', 'v7_eah_sweet_g'),
    ('meal_g', 'v1_meal_g', 'v7_meal_g'),
    ('eah_g', 'v1_eah_g', 'v7_eah_g')
]

# Initialize an empty DataFrame to store the t-test results
ttest_results = pd.DataFrame(columns=['Variable Pair', 'T-Statistic', 'P-Value'])

# Perform paired t-tests for each variable pair
for var_name, col_v1, col_v7 in variables_to_compare:
    t_statistic, p_value = ttest_rel(df[col_v1], df[col_v7])
    formatted_p_value = "{:.3f}".format(p_value)  # Format p-value to two decimal places
    ttest_results.loc[len(ttest_results)] = [var_name, t_statistic, formatted_p_value]

print(ttest_results)

# Liking ratings

 each food item liking rating at v1 and v7 , Independent T test for each food item comparing between v1 and v7

In [ ]:
def analyze_food_items(df):
    # Extracting the column names for v1 and v7 time points
    v1_columns = [col for col in df.columns if col.startswith("v1_vas_")]
    v7_columns = [col for col in df.columns if col.startswith("v7_vas_")]

    # Initializing a list to store results
    results = []

    # Looping through each pair of columns
    for v1_col, v7_col in zip(v1_columns, v7_columns):
        food_item = v1_col.split("v1_vas_")[-1]  # Extracting the food item name

        # Calculating means and standard errors
        mean_v1 = df[v1_col].mean()
        se_v1 = df[v1_col].sem()
        mean_v7 = df[v7_col].mean()
        se_v7 = df[v7_col].sem()

        # Performing t-test
        t_stat, p_value = stats.ttest_ind(df[v1_col].dropna(), df[v7_col].dropna())

        # Storing results
        results.append({
            "Food Item": food_item,
            "Mean V1": mean_v1,
            "SE V1": se_v1,
            "Mean V7": mean_v7,
            "SE V7": se_v7,
            "T-Statistic": t_stat,
            "P-Value": p_value
        })

    # Converting results to DataFrame for better visualization
    results_df = pd.DataFrame(results)
    return results_df

# Set display option for floating point numbers
pd.set_option('display.float_format', lambda x: '%.10f' % x)

# Analyzing the DataFrame
results = analyze_food_items(df)

# Printing the results
print(results)

Savory and sweet categories comparing between v1 vs v7

In [ ]:
# Define the savory and sweet items
savory_items = ['cornchip', 'pretzle', 'popcorn']
sweet_items = ['cookie', 'icecream', 'hershey', 'starburst', 'skittle', 'brownie']

def analyze_categories(df, savory_items, sweet_items):
    # Initialize results
    results = []

    # Function to calculate mean, SE, and t-test for a category
    def analyze_category(category_name, item_list):
        # Creating combined columns for v1 and v7
        df[f'v1_{category_name}'] = df[[f'v1_vas_{item}' for item in item_list]].mean(axis=1)
        df[f'v7_{category_name}'] = df[[f'v7_vas_{item}' for item in item_list]].mean(axis=1)

        # Calculating means and standard errors
        mean_v1 = df[f'v1_{category_name}'].mean()
        se_v1 = df[f'v1_{category_name}'].sem()
        mean_v7 = df[f'v7_{category_name}'].mean()
        se_v7 = df[f'v7_{category_name}'].sem()

        # Performing t-test
        t_stat, p_value = stats.ttest_ind(df[f'v1_{category_name}'].dropna(), df[f'v7_{category_name}'].dropna())

        return {
            "Category": category_name,
            "Mean V1": mean_v1,
            "SE V1": se_v1,
            "Mean V7": mean_v7,
            "SE V7": se_v7,
            "T-Statistic": t_stat,
            "P-Value": p_value
        }

    # Analyzing Savory category
    results.append(analyze_category("Savory", savory_items))

    # Analyzing Sweet category
    results.append(analyze_category("Sweet", sweet_items))

    # Converting results to DataFrame for better visualization
    results_df = pd.DataFrame(results)
    return results_df

# Set display option for floating point numbers
pd.set_option('display.float_format', lambda x: '%.10f' % x)


# Example usage:
results = analyze_categories(df, savory_items, sweet_items)
print(results)

Savory vs sweet categories at v1 AND v7

In [ ]:
import pandas as pd
from scipy import stats

def compare_sweet_savory(df, savory_items, sweet_items):
    # Function to calculate the mean and SE for a category at a given time point
    def category_stats(df, item_list, time_point):
        category_means = df[[f'{time_point}_vas_{item}' for item in item_list]].mean(axis=1)
        mean = category_means.mean()
        se = category_means.sem()
        return mean, se, category_means

    # Calculate category means and SE for both time points
    savory_mean_v1, savory_se_v1, savory_v1 = category_stats(df, savory_items, 'v1')
    sweet_mean_v1, sweet_se_v1, sweet_v1 = category_stats(df, sweet_items, 'v1')
    savory_mean_v7, savory_se_v7, savory_v7 = category_stats(df, savory_items, 'v7')
    sweet_mean_v7, sweet_se_v7, sweet_v7 = category_stats(df, sweet_items, 'v7')

    # Perform t-tests for sweet vs savory at v1 and v7 with sweet as the first group
    t_stat_v1, p_value_v1 = stats.ttest_ind(sweet_v1.dropna(), savory_v1.dropna())
    t_stat_v7, p_value_v7 = stats.ttest_ind(sweet_v7.dropna(), savory_v7.dropna())

    # Organize the results
    results = pd.DataFrame({
        "Category": ["Sweet", "Savory", "Sweet", "Savory"],
        "Time Point": ["V1", "V1", "V7", "V7"],
        "Mean": [sweet_mean_v1, savory_mean_v1, sweet_mean_v7, savory_mean_v7],
        "SE": [sweet_se_v1, savory_se_v1, sweet_se_v7, savory_se_v7],
        "T-Statistic": [t_stat_v1, None, t_stat_v7, None],
        "P-Value": [p_value_v1, None, p_value_v7, None]
    })

    return results

# Set display option for floating point numbers
pd.set_option('display.float_format', lambda x: '%.10f' % x)

# Define the savory and sweet items
savory_items = ['cornchip', 'pretzle', 'popcorn']
sweet_items = ['cookie', 'icecream', 'hershey', 'starburst', 'skittle', 'brownie']


results = compare_sweet_savory(df, savory_items, sweet_items)
print(results)


In [ ]:
import pandas as pd
from scipy import stats

def compare_sweet_savory(df, savory_items, sweet_items):
    # Function to calculate the mean for a category at a given time point
    def category_mean(df, item_list, time_point):
        # Creating a column for the category mean at the specified time point
        category_col = [f'{time_point}_vas_{item}' for item in item_list]
        df[f'{time_point}_category_mean'] = df[category_col].mean(axis=1)
        return df[f'{time_point}_category_mean']

    # Calculate category means for both time points
    savory_mean_v1 = category_mean(df, savory_items, 'v1')
    sweet_mean_v1 = category_mean(df, sweet_items, 'v1')
    savory_mean_v7 = category_mean(df, savory_items, 'v7')
    sweet_mean_v7 = category_mean(df, sweet_items, 'v7')

    # Perform t-tests
    t_stat_v1, p_value_v1 = stats.ttest_ind(savory_mean_v1.dropna(), sweet_mean_v1.dropna())
    t_stat_v7, p_value_v7 = stats.ttest_ind(savory_mean_v7.dropna(), sweet_mean_v7.dropna())

    # Results
    return pd.DataFrame({
        "Time Point": ["v1", "v7"],
        "T-Statistic": [t_stat_v1, t_stat_v7],
        "P-Value": [p_value_v1, p_value_v7]
    })

# Set display option for floating point numbers
pd.set_option('display.float_format', lambda x: '%.10f' % x)

# Define the savory and sweet items
savory_items = ['cornchip', 'pretzle', 'popcorn']
sweet_items = ['cookie', 'icecream', 'hershey', 'starburst', 'skittle', 'brownie']

# Example usage:
results = compare_sweet_savory(df, savory_items, sweet_items)
print(results)

## Analysis for EAH kcal as predictor of FMI with meal kcal intake as an additional covariate

### model for T1 EAH-kcal vs T2 FMI with meal_kcal controlled

### model for T1 EAH-kcal vs T2 FMI with meal_kcal additionally controlled
### covariates: baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1, main: predictor : T1 EAH kcal

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHkcalFMImealkcal = df_linear_reg[['v1_FMI','sex', 'risk_status_mom', 'income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff','v1_meal_kcal', 'v1_eah_kcal']]
y_EAHkcalFMImealkcal= df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHkcalFMImealkcal= sm.add_constant(X_EAHkcalFMImealkcal)

# Fit a linear regression model using ols
mod_EAHkcalFMImealkcal = sm.OLS(y_EAHkcalFMImealkcal, X_EAHkcalFMImealkcal).fit()

# Get summary of the regression model
print(mod_EAHkcalFMImealkcal.summary())

model assumption check

In [ ]:
# checking for model assumptions
# Calculate the residuals
from scipy.stats import shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
residuals_EAHkcalFMImealkcal= mod_EAHkcalFMImealkcal.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHkcalFMImealkcal = durbin_watson(residuals_EAHkcalFMImealkcal)
print("Durbin-Watson Statistic:", dw_statistic_EAHkcalFMImealkcal)
if 0 < dw_statistic_EAHkcalFMImealkcal< 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHkcalFMImealkcal.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHkcalFMImealkcal.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHkcalFMImealkcal.values, i) for i in range(X_EAHkcalFMImealkcal.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHkcalFMImealkcal.fittedvalues, y=residuals_EAHkcalFMImealkcal)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()


bp_test = het_breuschpagan(residuals_EAHkcalFMImealkcal,  mod_EAHkcalFMImealkcal.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)") 

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHkcalFMImealkcal, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHkcalFMImealkcal, p_value_EAHkcalFMImealkcal = shapiro(residuals_EAHkcalFMImealkcal)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHkcalFMImealkcal)
print("p-value:", p_value_EAHkcalFMImealkcal)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHkcalFMImealkcal > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### model for T1 EAH-gram vs T2 FMI with meal_gram additionally controlled
### covariates: baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1, main: predictor : T1 EAH gram

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHgFMImealg = df_linear_reg[['v1_FMI','sex', 'risk_status_mom', 'income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff','v1_meal_g', 'v1_eah_g']]
y_EAHgFMImealg= df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHgFMImealg= sm.add_constant(X_EAHgFMImealg)

# Fit a linear regression model using ols
mod_EAHgFMImealg = sm.OLS(y_EAHgFMImealg, X_EAHgFMImealg).fit()

# Get summary of the regression model
print(mod_EAHgFMImealg.summary())

model assumption check

In [ ]:
# checking for model assumptions
# Calculate the residuals
from scipy.stats import shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
residuals_EAHgFMImealg= mod_EAHgFMImealg.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHgFMImealg = durbin_watson(residuals_EAHgFMImealg)
print("Durbin-Watson Statistic:", dw_statistic_EAHgFMImealg)
if 0 < dw_statistic_EAHgFMImealg< 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHgFMImealg.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHgFMImealg.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHgFMImealg.values, i) for i in range(X_EAHgFMImealg.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHgFMImealg.fittedvalues, y=residuals_EAHgFMImealg)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHgFMImealg,  mod_EAHgFMImealg.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)") 

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHgFMImealg, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHgFMImealg, p_value_EAHgFMImealg = shapiro(residuals_EAHgFMImealg)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHgFMImealg)
print("p-value:", p_value_EAHgFMImealg)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHgFMImealg > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

SE reverse-scaling for reporting

In [ ]:
# Calculate the scaled standard error for intake values
sd_meal_g = df['v1_meal_g'].std()
sd_eah_g = df['v1_eah_g'].std()
sd_eah_sweet_g = df['v1_eah_sweet_g'].std()
sd_eah_sav_g = df['v1_eah_sav_g'].std()

sd_meal_kcal = df['v1_meal_kcal'].std()
sd_eah_kcal = df['v1_eah_kcal'].std()
sd_eah_sweet_kcal = df['v1_eah_sweet_kcal'].std()
sd_eah_sav_kcal = df['v1_eah_sav_kcal'].std()

sd_FMI = df['v7_FMI'].std()


scaled_se_eah_fmi_kcal_mealkcal = mod_EAHkcalFMImealkcal.bse['v1_eah_kcal'] * sd_FMI * (100 / sd_eah_kcal)
scaled_se_eah_fmi_g_mealg = mod_EAHgFMImealg.bse['v1_eah_g'] * sd_FMI * (100 / sd_eah_g)

print("Change in SE of FMI for 100 kcal increase in:")
print("SE of beta EAH kcal:{:.2f}".format(scaled_se_eah_fmi_kcal_mealkcal))


print("Change in SE of FMI for 100 g increase in:")

print("SE of beta EAH g:{:.2f}".format(scaled_se_eah_fmi_g_mealg))

In [ ]:
# Assuming df is your DataFrame
# Calculate standard deviations for necessary variables
sd_meal_g = df_linear_reg['v1_meal_g'].std()
sd_eah_g = df_linear_reg['v1_eah_g'].std()
sd_FMI = df_linear_reg['v7_FMI'].std()

# Define a function to compute the change in FMI
def change_in_FMI(coef, sd_FMI, sd_predictor):
    return coef * sd_FMI * (100 / sd_predictor)

# Select specific columns as predictors (X) and response (y) for gram analysis
X_EAHg = df_linear_reg[['v1_FMI', 'sex', 'risk_status_mom', 'income', 'parent_ed', 'v7_p_pds_imputed_2', 'v7_p_pds_imputed_3',  'age_diff', 'v1_meal_g', 'v1_eah_g']]
y_EAHg = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHg = sm.add_constant(X_EAHg)

# Fit a linear regression model using ols
mod_EAHg = sm.OLS(y_EAHg, X_EAHg).fit()

# Extract coefficient for 100g increase in EAH and calculate change in FMI
coef_eah_g = mod_EAHg.params['v1_eah_g']
change_eah_fmi_g = change_in_FMI(coef_eah_g, sd_FMI, sd_eah_g)

# Select specific columns as predictors (X) and response (y) for kcal analysis
X_EAHkcal = df_linear_reg[['v1_FMI', 'sex', 'risk_status_mom', 'income', 'parent_ed', 'v7_p_pds_imputed_2', 'v7_p_pds_imputed_3',  'age_diff', 'v1_meal_kcal', 'v1_eah_kcal']]
y_EAHkcal = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHkcal = sm.add_constant(X_EAHkcal)

# Fit a linear regression model using ols
mod_EAHkcal = sm.OLS(y_EAHkcal, X_EAHkcal).fit()

# Extract coefficient for 100kcal increase in EAH and calculate change in FMI
coef_eah_kcal = mod_EAHkcal.params['v1_eah_kcal']
change_eah_fmi_kcal = change_in_FMI(coef_eah_kcal, sd_FMI, sd_eah_g)

# Output the results
print("Change in FMI for 100 kcal increase in EAH (controlled for meal kcal): {:.2f}".format(change_eah_fmi_kcal))
print("Change in FMI for 100 g increase in EAH (controlled for meal g): {:.2f}".format(change_eah_fmi_g))

In [ ]:
# Assuming df_linear_reg is your DataFrame and the models mod_EAHkcal and mod_EAHg are correctly specified

# Create a 1x2 subplot layout
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Partial regression plot for EAH total (kcal) vs FMI
plot1 = sm.graphics.plot_partregress('v7_FMI', 'v1_eah_kcal', 
                                     ['v1_FMI', 'sex', 'risk_status_mom', 'income', 'parent_ed', 
                                      'v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff', 'v1_meal_kcal'], 
                                     data=df_linear_reg, obs_labels=False, ax=axes[0])
axes[0].set_xlabel("Baseline EAH Intake (kcal)")
axes[0].set_ylabel("Follow-up FMI score with T1 meal (kcal) controlled")
axes[0].set_title("Baseline Total EAH kcal Intake vs. Follow-up FMI")
axes[0].annotate(f"Partial Correlation: {mod_EAHkcal.params['v1_eah_kcal']:.3f}\n"
                 f"R-squared: {mod_EAHkcal.rsquared:.3f}\n"
                 f"P-value: {mod_EAHkcal.pvalues['v1_eah_kcal']:.3f}\n"
                 f"Change in FMI for 100 kcal increase: {change_eah_fmi_kcal:.3f}", 
                 xy=(0.05, 0.8), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression plot for EAH total (gram) vs FMI
plot2 = sm.graphics.plot_partregress('v7_FMI', 'v1_eah_g', 
                                     ['v1_FMI', 'sex', 'risk_status_mom', 'income', 'parent_ed', 
                                      'v7_p_pds_imputed_2', 'v7_p_pds_imputed_3',  'age_diff', 'v1_meal_g'], 
                                     data=df_linear_reg, obs_labels=False, ax=axes[1])
axes[1].set_xlabel("Baseline EAH Intake (gram)")
axes[1].set_ylabel("Follow-up FMI score with T1 meal (gram) controlled")
axes[1].set_title("Baseline Total EAH gram Intake vs. Follow-up FMI")
axes[1].annotate(f"Partial Correlation: {mod_EAHg.params['v1_eah_g']:.3f}\n"
                 f"R-squared: {mod_EAHg.rsquared:.3f}\n"
                 f"P-value: {mod_EAHg.pvalues['v1_eah_g']:.3f}\n"
                 f"Change in FMI for 100 g increase: {change_eah_fmi_g:.3f}", 
                 xy=(0.05, 0.8), xycoords='axes fraction', fontsize=12, color='blue')

fig.tight_layout()
plt.suptitle("Partial Correlation Plots: Intake EAH Variables (kcal and grams) as Predictors of Follow-Up Adiposity (FMI) with meal intake controlled", fontsize=16)
plt.subplots_adjust(top=0.8)
plt.show()

# Liking ratings

 each food item liking rating at v1 and v7 , Independent T test for each food item comparing between v1 and v7

In [ ]:
import pandas as pd
from scipy import stats

def analyze_food_items(df):
    # Extracting the column names for v1 and v7 time points
    v1_columns = [col for col in df.columns if col.startswith("v1_vas_")]
    v7_columns = [col for col in df.columns if col.startswith("v7_vas_")]

    # Initializing a list to store results
    results = []

    # Looping through each pair of columns
    for v1_col, v7_col in zip(v1_columns, v7_columns):
        food_item = v1_col.split("v1_vas_")[-1]  # Extracting the food item name

        # Calculating means and standard errors
        mean_v1 = df[v1_col].mean()
        se_v1 = df[v1_col].sem()
        mean_v7 = df[v7_col].mean()
        se_v7 = df[v7_col].sem()

        # Performing t-test
        t_stat, p_value = stats.ttest_ind(df[v1_col].dropna(), df[v7_col].dropna())

        # Storing results
        results.append({
            "Food Item": food_item,
            "Mean V1": mean_v1,
            "SE V1": se_v1,
            "Mean V7": mean_v7,
            "SE V7": se_v7,
            "T-Statistic": t_stat,
            "P-Value": p_value
        })

    # Converting results to DataFrame for better visualization
    results_df = pd.DataFrame(results)
    return results_df

# Set display option for floating point numbers
pd.set_option('display.float_format', lambda x: '%.10f' % x)

# Analyzing the DataFrame
results = analyze_food_items(df)

# Printing the results
print(results)


Savory and sweet categories comparing between v1 vs v7

In [ ]:
# Define the savory and sweet items
savory_items = ['cornchip', 'pretzle', 'popcorn']
sweet_items = ['cookie', 'icecream', 'hershey', 'starburst', 'skittle', 'brownie']

def analyze_categories(df, savory_items, sweet_items):
    # Initialize results
    results = []

    # Function to calculate mean, SE, and t-test for a category
    def analyze_category(category_name, item_list):
        # Creating combined columns for v1 and v7
        df[f'v1_{category_name}'] = df[[f'v1_vas_{item}' for item in item_list]].mean(axis=1)
        df[f'v7_{category_name}'] = df[[f'v7_vas_{item}' for item in item_list]].mean(axis=1)

        # Calculating means and standard errors
        mean_v1 = df[f'v1_{category_name}'].mean()
        se_v1 = df[f'v1_{category_name}'].sem()
        mean_v7 = df[f'v7_{category_name}'].mean()
        se_v7 = df[f'v7_{category_name}'].sem()

        # Performing t-test
        t_stat, p_value = stats.ttest_ind(df[f'v1_{category_name}'].dropna(), df[f'v7_{category_name}'].dropna())

        return {
            "Category": category_name,
            "Mean V1": mean_v1,
            "SE V1": se_v1,
            "Mean V7": mean_v7,
            "SE V7": se_v7,
            "T-Statistic": t_stat,
            "P-Value": p_value
        }

    # Analyzing Savory category
    results.append(analyze_category("Savory", savory_items))

    # Analyzing Sweet category
    results.append(analyze_category("Sweet", sweet_items))

    # Converting results to DataFrame for better visualization
    results_df = pd.DataFrame(results)
    return results_df

# Set display option for floating point numbers
pd.set_option('display.float_format', lambda x: '%.10f' % x)


# Example usage:
results = analyze_categories(df, savory_items, sweet_items)
print(results)

Savory vs sweet categories at v1 AND v7

In [ ]:
import pandas as pd
from scipy import stats

def compare_sweet_savory(df, savory_items, sweet_items):
    # Function to calculate the mean and SE for a category at a given time point
    def category_stats(df, item_list, time_point):
        category_means = df[[f'{time_point}_vas_{item}' for item in item_list]].mean(axis=1)
        mean = category_means.mean()
        se = category_means.sem()
        return mean, se, category_means

    # Calculate category means and SE for both time points
    savory_mean_v1, savory_se_v1, savory_v1 = category_stats(df, savory_items, 'v1')
    sweet_mean_v1, sweet_se_v1, sweet_v1 = category_stats(df, sweet_items, 'v1')
    savory_mean_v7, savory_se_v7, savory_v7 = category_stats(df, savory_items, 'v7')
    sweet_mean_v7, sweet_se_v7, sweet_v7 = category_stats(df, sweet_items, 'v7')

    # Perform t-tests for sweet vs savory at v1 and v7 with sweet as the first group
    t_stat_v1, p_value_v1 = stats.ttest_ind(sweet_v1.dropna(), savory_v1.dropna())
    t_stat_v7, p_value_v7 = stats.ttest_ind(sweet_v7.dropna(), savory_v7.dropna())

    # Organize the results
    results = pd.DataFrame({
        "Category": ["Sweet", "Savory", "Sweet", "Savory"],
        "Time Point": ["V1", "V1", "V7", "V7"],
        "Mean": [sweet_mean_v1, savory_mean_v1, sweet_mean_v7, savory_mean_v7],
        "SE": [sweet_se_v1, savory_se_v1, sweet_se_v7, savory_se_v7],
        "T-Statistic": [t_stat_v1, None, t_stat_v7, None],
        "P-Value": [p_value_v1, None, p_value_v7, None]
    })

    return results

# Set display option for floating point numbers
pd.set_option('display.float_format', lambda x: '%.10f' % x)

# Define the savory and sweet items
savory_items = ['cornchip', 'pretzle', 'popcorn']
sweet_items = ['cookie', 'icecream', 'hershey', 'starburst', 'skittle', 'brownie']


results = compare_sweet_savory(df, savory_items, sweet_items)
print(results)


T-test for comparison between sweet vs savory at T1, and at T2

In [ ]:
import pandas as pd
from scipy import stats

def compare_sweet_savory(df, savory_items, sweet_items):
    # Function to calculate the mean for a category at a given time point
    def category_mean(df, item_list, time_point):
        # Creating a column for the category mean at the specified time point
        category_col = [f'{time_point}_vas_{item}' for item in item_list]
        df[f'{time_point}_category_mean'] = df[category_col].mean(axis=1)
        return df[f'{time_point}_category_mean']

    # Calculate category means for both time points
    savory_mean_v1 = category_mean(df, savory_items, 'v1')
    sweet_mean_v1 = category_mean(df, sweet_items, 'v1')
    savory_mean_v7 = category_mean(df, savory_items, 'v7')
    sweet_mean_v7 = category_mean(df, sweet_items, 'v7')

    # Perform t-tests
    t_stat_v1, p_value_v1 = stats.ttest_ind(savory_mean_v1.dropna(), sweet_mean_v1.dropna())
    t_stat_v7, p_value_v7 = stats.ttest_ind(savory_mean_v7.dropna(), sweet_mean_v7.dropna())

    # Results
    return pd.DataFrame({
        "Time Point": ["v1", "v7"],
        "T-Statistic": [t_stat_v1, t_stat_v7],
        "P-Value": [p_value_v1, p_value_v7]
    })

# Set display option for floating point numbers
pd.set_option('display.float_format', lambda x: '%.10f' % x)

# Define the savory and sweet items
savory_items = ['cornchip', 'pretzle', 'popcorn']
sweet_items = ['cookie', 'icecream', 'hershey', 'starburst', 'skittle', 'brownie']

# Example usage:
results = compare_sweet_savory(df, savory_items, sweet_items)
print(results)

## additional analyses

### model fo T1 EAH-kcal vs T2 FMI with meal_kcal controlled

### covariates: baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1,T1 meal kcal, main predictor : T1 EAH kcal

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHkcalFMImealkcal = df_linear_reg[['v1_FMI','sex', 'risk_status_mom', 'income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','v1_meal_kcal', 'v1_eah_kcal']]
y_EAHkcalFMImealkcal= df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHkcalFMImealkcal= sm.add_constant(X_EAHkcalFMImealkcal)

# Fit a linear regression model using ols
mod_EAHkcalFMImealkcal = sm.OLS(y_EAHkcalFMImealkcal, X_EAHkcalFMImealkcal).fit()

# Get summary of the regression model
print(mod_EAHkcalFMImealkcal.summary())

model assumption check

In [ ]:
# checking for model assumptions
# Calculate the residuals
from scipy.stats import shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
residuals_EAHkcalFMImealkcal= mod_EAHkcalFMImealkcal.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHkcalFMImealkcal = durbin_watson(residuals_EAHkcalFMImealkcal)
print("Durbin-Watson Statistic:", dw_statistic_EAHkcalFMImealkcal)
if 0 < dw_statistic_EAHkcalFMImealkcal< 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHkcalFMImealkcal.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHkcalFMImealkcal.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHkcalFMImealkcal.values, i) for i in range(X_EAHkcalFMImealkcal.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHkcalFMImealkcal.fittedvalues, y=residuals_EAHkcalFMImealkcal)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()


bp_test = het_breuschpagan(residuals_EAHkcalFMImealkcal,  mod_EAHkcalFMImealkcal.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)") 

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHkcalFMImealkcal, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHkcalFMImealkcal, p_value_EAHkcalFMImealkcal = shapiro(residuals_EAHkcalFMImealkcal)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHkcalFMImealkcal)
print("p-value:", p_value_EAHkcalFMImealkcal)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHkcalFMImealkcal > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### model for T1 EAH-gram vs T2 FMI with meal_gram controlled

### covariates: baseline FMI, * sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1, T1 meal gram, main predictor : T1 EAH gram

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHgFMImealg = df_linear_reg[['v1_FMI','sex', 'risk_status_mom', 'income','parent_ed','v7_p_pds_imputed_2', 'v7_p_pds_imputed_3','age_diff','v1_meal_g', 'v1_eah_g']]
y_EAHgFMImealg= df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHgFMImealg= sm.add_constant(X_EAHgFMImealg)

# Fit a linear regression model using ols
mod_EAHgFMImealg = sm.OLS(y_EAHgFMImealg, X_EAHgFMImealg).fit()

# Get summary of the regression model
print(mod_EAHgFMImealg.summary())

model assumption check

In [ ]:
# checking for model assumptions
# Calculate the residuals
from scipy.stats import shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
residuals_EAHgFMImealg= mod_EAHgFMImealg.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHgFMImealg = durbin_watson(residuals_EAHgFMImealg)
print("Durbin-Watson Statistic:", dw_statistic_EAHgFMImealg)
if 0 < dw_statistic_EAHgFMImealg< 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHgFMImealg.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHgFMImealg.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHgFMImealg.values, i) for i in range(X_EAHgFMImealg.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHgFMImealg.fittedvalues, y=residuals_EAHgFMImealg)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHgFMImealg,  mod_EAHgFMImealg.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)") 

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHgFMImealg, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHgFMImealg, p_value_EAHgFMImealg = shapiro(residuals_EAHgFMImealg)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHgFMImealg)
print("p-value:", p_value_EAHgFMImealg)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHgFMImealg > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

SE-reverse scaling for reporting 

In [ ]:
# Calculate the scaled standard error for intake values
sd_meal_g = df['v1_meal_g'].std()
sd_eah_g = df['v1_eah_g'].std()
sd_eah_sweet_g = df['v1_eah_sweet_g'].std()
sd_eah_sav_g = df['v1_eah_sav_g'].std()

sd_meal_kcal = df['v1_meal_kcal'].std()
sd_eah_kcal = df['v1_eah_kcal'].std()
sd_eah_sweet_kcal = df['v1_eah_sweet_kcal'].std()
sd_eah_sav_kcal = df['v1_eah_sav_kcal'].std()

sd_FMI = df['v7_FMI'].std()


scaled_se_eah_fmi_kcal_mealkcal = mod_EAHkcalFMImealkcal.bse['v1_eah_kcal'] * sd_FMI * (100 / sd_eah_kcal)
scaled_se_eah_fmi_g_mealg = mod_EAHgFMImealg.bse['v1_eah_g'] * sd_FMI * (100 / sd_eah_g)

print("Change in SE of FMI for 100 kcal increase in:")
print("SE of beta EAH kcal:{:.2f}".format(scaled_se_eah_fmi_kcal_mealkcal))


print("Change in SE of FMI for 100 g increase in:")

print("SE of beta EAH g:{:.2f}".format(scaled_se_eah_fmi_g_mealg))

In [ ]:
import pandas as pd
import statsmodels.api as sm
# Assuming df is your DataFrame
# Calculate standard deviations for necessary variables
sd_meal_g = df_linear_reg['v1_meal_g'].std()
sd_eah_g = df_linear_reg['v1_eah_g'].std()
sd_FMI = df_linear_reg['v7_FMI'].std()

# Define a function to compute the change in FMI
def change_in_FMI(coef, sd_FMI, sd_predictor):
    return coef * sd_FMI * (100 / sd_predictor)

# Select specific columns as predictors (X) and response (y) for gram analysis
X_EAHg = df_linear_reg[['v1_FMI', 'sex', 'risk_status_mom', 'income', 'parent_ed', 'v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff', 'v1_meal_g', 'v1_eah_g']]
y_EAHg = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHg = sm.add_constant(X_EAHg)

# Fit a linear regression model using ols
mod_EAHg = sm.OLS(y_EAHg, X_EAHg).fit()

# Extract coefficient for 100g increase in EAH and calculate change in FMI
coef_eah_g = mod_EAHg.params['v1_eah_g']
change_eah_fmi_g = change_in_FMI(coef_eah_g, sd_FMI, sd_eah_g)

# Select specific columns as predictors (X) and response (y) for kcal analysis
X_EAHkcal = df_linear_reg[['v1_FMI', 'sex', 'risk_status_mom', 'income', 'parent_ed', 'v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff', 'v1_meal_kcal', 'v1_eah_kcal']]
y_EAHkcal = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHkcal = sm.add_constant(X_EAHkcal)

# Fit a linear regression model using ols
mod_EAHkcal = sm.OLS(y_EAHkcal, X_EAHkcal).fit()

# Extract coefficient for 100kcal increase in EAH and calculate change in FMI
coef_eah_kcal = mod_EAHkcal.params['v1_eah_kcal']
change_eah_fmi_kcal = change_in_FMI(coef_eah_kcal, sd_FMI, sd_eah_g)

# Output the results
print("Change in FMI for 100 kcal increase in EAH (controlled for meal kcal): {:.2f}".format(change_eah_fmi_kcal))
print("Change in FMI for 100 g increase in EAH (controlled for meal g): {:.2f}".format(change_eah_fmi_g))


In [ ]:
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Assuming df_linear_reg is your DataFrame and the models mod_EAHkcal and mod_EAHg are correctly specified

# Create a 1x2 subplot layout
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Partial regression plot for EAH total (kcal) vs FMI
plot1 = sm.graphics.plot_partregress('v7_FMI', 'v1_eah_kcal', 
                                     ['v1_FMI', 'sex', 'risk_status_mom', 'income', 'parent_ed', 
                                      'v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff', 'v1_meal_kcal'], 
                                     data=df_linear_reg, obs_labels=False, ax=axes[0])
axes[0].set_xlabel("Baseline EAH Intake (kcal)")
axes[0].set_ylabel("Follow-up FMI score with T1 meal (kcal) controlled")
axes[0].set_title("Baseline Total EAH kcal Intake vs. Follow-up FMI")
axes[0].annotate(f"Partial Correlation: {mod_EAHkcal.params['v1_eah_kcal']:.3f}\n"
                 f"R-squared: {mod_EAHkcal.rsquared:.3f}\n"
                 f"P-value: {mod_EAHkcal.pvalues['v1_eah_kcal']:.3f}\n"
                 f"Change in FMI for 100 kcal increase: {change_eah_fmi_kcal:.3f}", 
                 xy=(0.05, 0.8), xycoords='axes fraction', fontsize=12, color='blue')

# Partial regression plot for EAH total (gram) vs FMI
plot2 = sm.graphics.plot_partregress('v7_FMI', 'v1_eah_g', 
                                     ['v1_FMI', 'sex', 'risk_status_mom', 'income', 'parent_ed', 
                                      'v7_p_pds_imputed_2', 'v7_p_pds_imputed_3', 'age_diff', 'v1_meal_g'], 
                                     data=df_linear_reg, obs_labels=False, ax=axes[1])
axes[1].set_xlabel("Baseline EAH Intake (gram)")
axes[1].set_ylabel("Follow-up FMI score with T1 meal (gram) controlled")
axes[1].set_title("Baseline Total EAH gram Intake vs. Follow-up FMI")
axes[1].annotate(f"Partial Correlation: {mod_EAHg.params['v1_eah_g']:.3f}\n"
                 f"R-squared: {mod_EAHg.rsquared:.3f}\n"
                 f"P-value: {mod_EAHg.pvalues['v1_eah_g']:.3f}\n"
                 f"Change in FMI for 100 g increase: {change_eah_fmi_g:.3f}", 
                 xy=(0.05, 0.8), xycoords='axes fraction', fontsize=12, color='blue')

fig.tight_layout()
plt.suptitle("Partial Correlation Plots: Intake EAH Variables (kcal and grams) as Predictors of Follow-Up Adiposity (FMI) with meal intake controlled", fontsize=16)
plt.subplots_adjust(top=0.8)
plt.show()

# ANALYSES BASED ON SANDWICH TYPE ON EAH INTAKE vs FMI

## checking if meal type (sweet and savory) has an effect on EAH intake

### EAH savory gram vs FMI with savory sandwich at pre-meal (gram)

#### covariates: baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1, meal sav sandwich gram, main: predictor : T1 EAH sav gram

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavgFMImealsavg = df_linear_reg[['v1_FMI','sex', 'risk_status_mom', 'income','parent_ed','v7_p_pds_imputed_2','v7_p_pds_imputed_3','age_diff','v1_sandw_sav_g', 'v1_eah_sav_g']]
y_EAHsavgFMImealsavg = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHsavgFMImealsavg = sm.add_constant(X_EAHsavgFMImealsavg)

# Fit a linear regression model using ols
mod_EAHsavgFMImealsavg = sm.OLS(y_EAHsavgFMImealsavg, X_EAHsavgFMImealsavg).fit()

# Get summary of the regression model
print(mod_EAHsavgFMImealsavg.summary())

model assumption

In [ ]:
# checking for model assumptions
# Calculate the residuals
from scipy.stats import shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
residuals_EAHsavgFMImealsavg = mod_EAHsavgFMImealsavg.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavgFMImealsavg = durbin_watson(residuals_EAHsavgFMImealsavg)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavgFMImealsavg)
if 0 < dw_statistic_EAHsavgFMImealsavg < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavgFMImealsavg.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavgFMImealsavg.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavgFMImealsavg.values, i) for i in range(X_EAHsavgFMImealsavg.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavgFMImealsavg.fittedvalues, y=residuals_EAHsavgFMImealsavg)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsavgFMImealsavg,  mod_EAHsavgFMImealsavg.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)") 

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavgFMImealsavg, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavgFMImealsavg, p_value_EAHsavgFMImealsavg = shapiro(residuals_EAHsavgFMImealsavg)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavgFMImealsavg)
print("p-value:", p_value_EAHsavgFMImealsavg)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsavgFMImealsavg > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### EAH sav kcal vs FMI with savory sandwich meal (kcal)  as a covariate

#### covariates: baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1, meal sav sandwich kcal, main: predictor : T1 EAH sav kcal

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsavkcalFMImealsavkcal = df_linear_reg[['v1_FMI','sex', 'risk_status_mom', 'income','parent_ed','v7_p_pds_imputed_2','v7_p_pds_imputed_3','age_diff','v1_sandw_sav_kcal', 'v1_eah_sav_kcal']]
y_EAHsavkcalFMImealsavkcal = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHsavkcalFMImealsavkcal = sm.add_constant(X_EAHsavkcalFMImealsavkcal)

# Fit a linear regression model using ols
mod_EAHsavkcalFMImealsavkcal = sm.OLS(y_EAHsavkcalFMImealsavkcal, X_EAHsavkcalFMImealsavkcal).fit()

# Get summary of the regression model
print(mod_EAHsavkcalFMImealsavkcal.summary())

model assumption check

In [ ]:
# checking for model assumptions
# Calculate the residuals
from scipy.stats import shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
residuals_EAHsavkcalFMImealsavkcal = mod_EAHsavkcalFMImealsavkcal.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsavkcalFMImealsavkcal = durbin_watson(residuals_EAHsavkcalFMImealsavkcal)
print("Durbin-Watson Statistic:", dw_statistic_EAHsavkcalFMImealsavkcal)
if 0 < dw_statistic_EAHsavkcalFMImealsavkcal < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsavkcalFMImealsavkcal.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsavkcalFMImealsavkcal.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsavkcalFMImealsavkcal.values, i) for i in range(X_EAHsavkcalFMImealsavkcal.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsavkcalFMImealsavkcal.fittedvalues, y=residuals_EAHsavkcalFMImealsavkcal)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsavkcalFMImealsavkcal,  mod_EAHsavkcalFMImealsavkcal.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)") 

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsavkcalFMImealsavkcal, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsavkcalFMImealsavkcal, p_value_EAHsavkcalFMImealsavkcal = shapiro(residuals_EAHsavkcalFMImealsavkcal)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsavkcalFMImealsavkcal)
print("p-value:", p_value_EAHsavkcalFMImealsavkcal)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsavkcalFMImealsavkcal > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### EAH sweet gram vs FMI with sweet meal (gram)

#### covariates: baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1, meal sweet sandwich gram, main: predictor : T1 EAH sweet gram

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetgFMImealsweetg = df_linear_reg[['v1_FMI','sex', 'risk_status_mom', 'income','parent_ed','v7_p_pds_imputed_2','v7_p_pds_imputed_3','age_diff','v1_consumed_pbj_sndwch_g','v1_eah_sweet_g']]
y_EAHsweetgFMImealsweetg = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHsweetgFMImealsweetg = sm.add_constant(X_EAHsweetgFMImealsweetg)

# Fit a linear regression model using ols
mod_EAHsweetgFMImealsweetg = sm.OLS(y_EAHsweetgFMImealsweetg, X_EAHsweetgFMImealsweetg).fit()

# Get summary of the regression model
print(mod_EAHsweetgFMImealsweetg.summary())

model assumption check

In [ ]:
# checking for model assumptions
# Calculate the residuals
from scipy.stats import shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
residuals_EAHsweetgFMImealsweetg = mod_EAHsweetgFMImealsweetg.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetgFMImealsweetg = durbin_watson(residuals_EAHsweetgFMImealsweetg)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetgFMImealsweetg)
if 0 < dw_statistic_EAHsweetgFMImealsweetg < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetgFMImealsweetg.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetgFMImealsweetg.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetgFMImealsweetg.values, i) for i in range(X_EAHsweetgFMImealsweetg.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetgFMImealsweetg.fittedvalues, y=residuals_EAHsweetgFMImealsweetg)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetgFMImealsweetg,  mod_EAHsweetgFMImealsweetg.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)") 

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetgFMImealsweetg, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetgFMImealsweetg, p_value_EAHsweetgFMImealsweetg = shapiro(residuals_EAHsweetgFMImealsweetg)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetgFMImealsweetg)
print("p-value:", p_value_EAHsweetgFMImealsweetg)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetgFMImealsweetg > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### EAH sweet kcal vs FMI with sweet sandwich meal kcal controlled

#### covariates: baseline FMI, sex, risk status based on maternal BMI, tanner stage category at T2, parent education, parent yearly income, age differences between v7 and v1, meal sweet sandwich kcal, main: predictor : T1 EAH sweet kcal

In [ ]:
# Select specific columns as predictors (X) and response (y)
X_EAHsweetkcalFMImealsweetkcal = df_linear_reg[['v1_FMI','sex', 'risk_status_mom', 'income','parent_ed','v7_p_pds_imputed_2','v7_p_pds_imputed_3','age_diff','v1_consumed_pbj_sndwch_kcal', 'v1_eah_sweet_kcal']]
y_EAHsweetkcalFMImealsweetkcal = df_linear_reg['v7_FMI']

# Add a constant term to the predictors (intercept)
X_EAHsweetkcalFMImealsweetkcal = sm.add_constant(X_EAHsweetkcalFMImealsweetkcal)

# Fit a linear regression model using ols
mod_EAHsweetkcalFMImealsweetkcal = sm.OLS(y_EAHsweetkcalFMImealsweetkcal, X_EAHsweetkcalFMImealsweetkcal).fit()

# Get summary of the regression model
print(mod_EAHsweetkcalFMImealsweetkcal.summary())

model assumption check

In [ ]:
# checking for model assumptions
# Calculate the residuals
from scipy.stats import shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
residuals_EAHsweetkcalFMImealsweetkcal = mod_EAHsweetkcalFMImealsweetkcal.resid

# 1. Linearity
# partial regression plots - refer later

# 2. Multicollinearity
print("MULTICOLLINEARITY and AUTOCORRELATION")
dw_statistic_EAHsweetkcalFMImealsweetkcal = durbin_watson(residuals_EAHsweetkcalFMImealsweetkcal)
print("Durbin-Watson Statistic:", dw_statistic_EAHsweetkcalFMImealsweetkcal)
if 0 < dw_statistic_EAHsweetkcalFMImealsweetkcal < 4:
    print("The Durbin-Watson statistic suggests no significant autocorrelation.")
else:
    print("The Durbin-Watson statistic indicates potential autocorrelation.")

correlation_matrix = X_EAHsweetkcalFMImealsweetkcal.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Matrix")
plt.show()


# Calculate VIF for each predictor variable
vif_data = pd.DataFrame()
vif_data["Variable"] = X_EAHsweetkcalFMImealsweetkcal.columns
vif_data["VIF"] = [variance_inflation_factor(X_EAHsweetkcalFMImealsweetkcal.values, i) for i in range(X_EAHsweetkcalFMImealsweetkcal.shape[1])]
print(vif_data)

# 3. Heteroskedasticity Assumption
print("CHECKING FOR HETEROSKEDASTICITY")
sns.scatterplot(x=mod_EAHsweetkcalFMImealsweetkcal.fittedvalues, y=residuals_EAHsweetkcalFMImealsweetkcal)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted Values Plot")
plt.show()

bp_test = het_breuschpagan(residuals_EAHsweetkcalFMImealsweetkcal,  mod_EAHsweetkcalFMImealsweetkcal.model.exog)
bp_statistic, bp_pvalue, _, _ = bp_test
print(f'Breusch-Pagan test statistic: {bp_statistic}, p-value: {bp_pvalue}')

if bp_pvalue < 0.05:
    print("The Breusch-Pagan test suggests heteroskedasticity (reject null hypothesis of homoskedasticity)")
else:
    print("The Breusch-Pagan test does not suggest heteroskedasticity (fail to reject null hypothesis of homoskedasticity)") 

# 4. Normal Distribution of Error Terms Assumption
# Create a Q-Q plot
print("NORMALITY")
stats.probplot(residuals_EAHsweetkcalFMImealsweetkcal, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()

# Perform Shapiro-Wilk test for normality
statistic_EAHsweetkcalFMImealsweetkcal, p_value_EAHsweetkcalFMImealsweetkcal = shapiro(residuals_EAHsweetkcalFMImealsweetkcal)

# Print the test results
print("Shapiro-Wilk Test Results:")
print("Statistic:", statistic_EAHsweetkcalFMImealsweetkcal)
print("p-value:", p_value_EAHsweetkcalFMImealsweetkcal)

# Check the p-value to determine whether residuals are normally distributed
alpha = 0.05  # Significance level
if p_value_EAHsweetkcalFMImealsweetkcal > alpha:
    print("Residuals are normally distributed (fail to reject null hypothesis)")
else:
    print("Residuals are not normally distributed (reject null hypothesis)")

### Exporting table with OLS model params

In [ ]:
import pandas as pd
import statsmodels.api as sm

# List of model names
model_names = [
    "mod_mealkcalFMI", "mod_EAHkcalFMI", "mod_EAHsweetkcalFMI", "mod_EAHsavkcalFMI",
    "mod_mealkcalBMIz", "mod_EAHkcalBMIz", "mod_EAHsweetkcalBMIz", "mod_EAHsavkcalBMIz",
    "mod_mealgFMI", "mod_EAHgFMI", "mod_EAHsweetgFMI", "mod_EAHsavgFMI",
     "mod_mealgBMIz", "mod_EAHgBMIz", "mod_EAHsweetgBMIz", "mod_EAHsavgBMIz",
     "mod_mealkcalFMI_audio", "mod_EAHkcalFMI_audio", "mod_EAHsweetkcalFMI_audio", "mod_EAHsavkcalFMI_audio",
    "mod_mealkcalBMIz_audio", "mod_EAHkcalBMIz_audio", "mod_EAHsweetkcalBMIz_audio", "mod_EAHsavkcalBMIz_audio",
    "mod_mealgFMI_audio", "mod_EAHgFMI_audio", "mod_EAHsweetgFMI_audio", "mod_EAHsavgFMI_audio",
     "mod_mealgBMIz_audio", "mod_EAHgBMIz_audio", "mod_EAHsweetgBMIz_audio", "mod_EAHsavgBMIz_audio", 
     "mod_EAHkcalFMI_item", "mod_EAHsweetkcalFMI_item", "mod_EAHsavkcalFMI_item",
    "mod_EAHkcalBMIz_item", "mod_EAHsweetkcalBMIz_item", "mod_EAHsavkcalBMIz_item",
    "mod_EAHgFMI_item", "mod_EAHsweetgFMI_item", "mod_EAHsavgFMI_item",
     "mod_EAHgBMIz_item", "mod_EAHsweetgBMIz_item", "mod_EAHsavgBMIz_item",
     "mod_EAHkcalFMImealkcal","mod_EAHsweetgFMImealsweetg","mod_EAHsavgFMImealsavg",
     "mod_EAHgFMImealg", "mod_EAHsweetkcalFMImealsweetkcal","mod_EAHsavkcalFMImealsavkcal"
]

# List to store model information
model_info = []

# Loop through each model
for model_name in model_names:
    model = globals()[model_name]  # Get the model object using its name
    
    # Extract parameter for the last predictor
    last_predictor = model.params.index[-1]
    last_param = model.params[last_predictor]
    last_se = model.bse[last_predictor] 
    last_p_value = model.pvalues[last_predictor]
    last_t_value = model.tvalues[last_predictor]
    last_conf_int = model.conf_int().loc[last_predictor]

    # Create a dictionary to store information
    model_entry = {
        "Model Name": model_name,
        "Last Predictor": last_predictor,
        "Last Parameter": last_param,
        "SE": last_se,
        "Last P-value": last_p_value,
        "Last T-value": last_t_value,
        "Last CI 95%": last_conf_int
    }

    model_info.append(model_entry)

# Create a DataFrame from the list of model information
models_df = pd.DataFrame(model_info)

models_df.to_excel("regression model results.xlsx",index=False)

# SEX DIFFERENCES EXPLORATORY ANALYSIS

## Tanner Stage check

Chi-square test : analysis in Tanner stage differences between boys vs girls at T1, T2, Tanner stage difference between T2-T1

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

df_tanner=df.copy()

# Contingency table for pds_tanner_cat vs sex
contingency_table1 = pd.crosstab(df_tanner['pds_tanner_cat'], df_tanner['sex'])
chi2_1, p_1, dof_1, ex_1 = chi2_contingency(contingency_table1)

# Contingency table for v7_pds_tanner_imputed vs sex
contingency_table2 = pd.crosstab(df_tanner['v7_p_pds_imputed'], df_tanner['sex'])
chi2_2, p_2, dof_2, ex_2 = chi2_contingency(contingency_table2)

# Add a new column for the difference between v7_pds_tanner_imputed and pds_tanner_cat
df_tanner['difference'] = df_tanner['v7_p_pds_imputed'] - df_tanner['pds_tanner_cat']

# Contingency table for difference vs sex
contingency_table3 = pd.crosstab(df_tanner['difference'], df_tanner['sex'])
chi2_3, p_3, dof_3, ex_3 = chi2_contingency(contingency_table3)

# Print the contingency tables and their degrees of freedom
print("Contingency table for pds_tanner_cat vs sex:")
print(contingency_table1)
print(f"Degrees of Freedom: {dof_1}")

print("\nContingency table for v7_p_pds_imputed vs sex:")
print(contingency_table2)
print(f"Degrees of Freedom: {dof_2}")

print("\nContingency table for (v7_p_pds_imputed - pds_tanner_cat) vs sex:")
print(contingency_table3)
print(f"Degrees of Freedom: {dof_3}")

# Print the chi-square test results
print("\nChi-square test results:")

print("Chi-square test for pds_tanner_cat vs sex:")
print(f"Chi2: {chi2_1}, p-value: {p_1}")

print("\nChi-square test for v7_p_pds_imputed vs sex:")
print(f"Chi2: {chi2_2}, p-value: {p_2}")

print("\nChi-square test for (v7_p_pds_imputed - pds_tanner_cat) vs sex:")
print(f"Chi2: {chi2_3}, p-value: {p_3}")


Scatterplots between boys vs girls for T1 intake (meal, EAH; grams, kcal) vs T2 outcomes (FMI, BMI-z)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_intake = df.copy()

# Map sex to labels
df_intake['sex'] = df_intake['sex'].map({0: 'male', 1: 'female'})

# Define the EAH intake variables and meal variables
eah_intake_vars = [
    'v1_eah_kcal',
    'v1_eah_sweet_kcal',
    'v1_eah_sav_kcal',
    'v1_eah_g',
    'v1_eah_sweet_g',
    'v1_eah_sav_g',
    'v1_meal_kcal',
    'v1_meal_g'
]

# Function to create descriptive labels for the variables
def get_x_label(var):
    if 'eah' in var:
        base = "EAH"
    else:
        base = "Meal"
    
    if 'sweet' in var:
        type_ = "Sweet"
    elif 'sav' in var:
        type_ = "Savory"
    else:
        type_ = "Total"
        
    if 'kcal' in var:
        unit = "kcal"
    else:
        unit = "grams"
    
    return f'{base} intake ({type_}, {unit}) at T1'

# Plot 1: T1 intake vs. T2 Adiposity (FMI)
fig, axs = plt.subplots(3, 3, figsize=(18, 18))
fig.suptitle('Differences in sex for relationship between T1 intake and T2 Adiposity (FMI)', fontsize=20)
axs = axs.flatten()

for i, eah_var in enumerate(eah_intake_vars):
    sns.scatterplot(
        data=df_intake,
        x=eah_var,
        y='v7_FMI',
        hue='sex',
        ax=axs[i]
    )
    sns.regplot(
        data=df_intake[df_intake['sex'] == 'male'],
        x=eah_var,
        y='v7_FMI',
        ax=axs[i],
        scatter=False,
        color='blue',
        label='male'
    )
    sns.regplot(
        data=df_intake[df_intake['sex'] == 'female'],
        x=eah_var,
        y='v7_FMI',
        ax=axs[i],
        scatter=False,
        color='orange',
        label='female'
    )
    axs[i].set_title(f'{get_x_label(eah_var)} vs T2 Adiposity (FMI)', fontsize=14)
    axs[i].set_xlabel(get_x_label(eah_var), fontsize=14)
    axs[i].set_ylabel('Follow-up Adiposity (FMI)', fontsize=14)

# Remove any unused subplots
for j in range(i + 1, len(axs)):
    fig.delaxes(axs[j])

# Add panel label for Plot 1
fig.text(0.05, 0.95, 'A.', fontsize=22, fontweight='bold', ha='center', va='center')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# Plot 2: T1 intake vs. T2 Weight Status (BMI-z)
fig, axs = plt.subplots(3, 3, figsize=(18, 18))
fig.suptitle('Differences in sex for relationship between T1 intake and T2 Weight Status (BMI-z)', fontsize=20)
axs = axs.flatten()

for i, eah_var in enumerate(eah_intake_vars):
    sns.scatterplot(
        data=df_intake,
        x=eah_var,
        y='v7_bmi_z',
        hue='sex',
        ax=axs[i]
    )
    sns.regplot(
        data=df_intake[df_intake['sex'] == 'male'],
        x=eah_var,
        y='v7_bmi_z',
        ax=axs[i],
        scatter=False,
        color='blue',
        label='male'
    )
    sns.regplot(
        data=df_intake[df_intake['sex'] == 'female'],
        x=eah_var,
        y='v7_bmi_z',
        ax=axs[i],
        scatter=False,
        color='orange',
        label='female'
    )
    axs[i].set_title(f'{get_x_label(eah_var)} vs Follow-up BMI-z (T2)', fontsize=14)
    axs[i].set_xlabel(get_x_label(eah_var), fontsize=14)
    axs[i].set_ylabel('Follow-up BMI-z (T2)', fontsize=14)

# Remove any unused subplots
for j in range(i + 1, len(axs)):
    fig.delaxes(axs[j])

# Add panel label for Plot 2
fig.text(0.05, 0.95, 'B.', fontsize=22, fontweight='bold', ha='center', va='center')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


# T1 and T2 intake measures - sex differences

### Graphs between T1 and T2 for boys and girls

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
import pandas as pd

df_intake = df.copy()

# Map sex to labels if not already done
df_intake['sex'] = df_intake['sex'].map({0: 'male', 1: 'female'})

# Filter data for boys and girls
df_boys = df_intake[df_intake['sex'] == 'male']
df_girls = df_intake[df_intake['sex'] == 'female']

# List of variables to plot
variables = [
    ('v1_eah_g', 'v7_eah_g'),
    ('v1_eah_sweet_g', 'v7_eah_sweet_g'),
    ('v1_eah_sav_g', 'v7_eah_sav_g'),
    ('v1_eah_kcal', 'v7_eah_kcal'),
    ('v1_eah_sweet_kcal', 'v7_eah_sweet_kcal'),
    ('v1_eah_sav_kcal', 'v7_eah_sav_kcal'),
    ('v1_meal_g', 'v7_meal_g'),
    ('v1_meal_kcal', 'v7_meal_kcal')
]

# Helper function to format variable names for labels
def format_label(label):
    label = label.replace('v1', 'T1').replace('v7', 'T2')
    label = label.replace('eah', 'EAH').replace('_sav', '_savory')
    label = label.replace('_g', ' grams').replace('_kcal', ' Kcal')
    label = label.replace('_', ' ').title()
    label = label.replace('Eah', 'EAH')  # Ensure 'EAH' is in all caps
    return label

def plot_comparison(df, sex_label, panel_label):
    # Set up the figure and axes for the plots
    fig, axs = plt.subplots(3, 3, figsize=(18, 16))
    fig.suptitle(f'Differences in Intake between T1 and T2 for {sex_label.capitalize()}', fontsize=20)

    # Plot each pair of variables and perform t-tests
    for i, (v1, v7) in enumerate(variables):
        # Prepare data for plotting
        melted_df = pd.melt(df[['sex', v1, v7]], id_vars='sex', value_vars=[v1, v7], var_name='variable')

        # Plot Baseline (v1) and Follow-up (v7)
        sns.boxplot(data=melted_df, x='variable', y='value', ax=axs[i//3, i%3], palette='pastel')
        sns.stripplot(data=melted_df, x='variable', y='value', ax=axs[i//3, i%3], color='black', jitter=True, alpha=0.6)
        axs[i//3, i%3].set_title(f'Comparison of {format_label(v1)} and {format_label(v7)}', fontsize=13)
        axs[i//3, i%3].set_xlabel('Time Point',fontsize=14)
        axs[i//3, i%3].set_ylabel(format_label(v1).replace('T1', '').strip(),fontsize=14)

        # Perform paired t-test
        t_stat, p_value = ttest_rel(df[v1], df[v7], nan_policy='omit')

        # Display the t-test results
        axs[i//3, i%3].text(0.1, 0.9, f'T-statistic: {t_stat:.2f}', transform=axs[i//3, i%3].transAxes, fontsize=12)
        axs[i//3, i%3].text(0.1, 0.8, f'p-value: {p_value:.2f}', transform=axs[i//3, i%3].transAxes, fontsize=12)

    # Leave the last subplot blank
    axs[2, 2].axis('off')

    # Add panel label (A. for boys and B. for girls)
    fig.text(0.05, 0.95, panel_label, fontsize=22, fontweight='bold', ha='center', va='center')

    # Adjust layout and show the plot
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

# Plot for boys with panel label A.
plot_comparison(df_boys, 'boys', 'A.')

# Plot for girls with panel label B.
plot_comparison(df_girls, 'girls', 'B.')


## T1 and T2 intake measures by sex

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
import pandas as pd

df_intake = df.copy()

# Map sex to labels if not already done
df_intake['sex'] = df_intake['sex'].map({0: 'male', 1: 'female'})

# List of baseline and follow-up variables grouped
baseline_vars = [
    'v1_eah_g', 'v1_eah_sweet_g', 'v1_eah_sav_g',
    'v1_eah_kcal', 'v1_eah_sweet_kcal', 'v1_eah_sav_kcal',
    'v1_meal_g', 'v1_meal_kcal'
]

followup_vars = [
    'v7_eah_g', 'v7_eah_sweet_g', 'v7_eah_sav_g',
    'v7_eah_kcal', 'v7_eah_sweet_kcal', 'v7_eah_sav_kcal',
    'v7_meal_g', 'v7_meal_kcal'
]

# Function to create descriptive labels for the variables
def get_label(var):
    if 'sweet' in var:
        type_ = "Sweet"
    elif 'sav' in var:
        type_ = "Savory"
    else:
        type_ = "Total"
    
    if 'kcal' in var:
        unit = "kcal"
    else:
        unit = "grams"
    
    return f'{type_} intake ({unit})'

# Set up the figure and axes for baseline variables
fig, axs = plt.subplots(3, 3, figsize=(18, 16))
fig.suptitle('Baseline (T1) Intake Measures by Sex', fontsize=20)

for i, var in enumerate(baseline_vars):
    # Prepare data for plotting
    melted_df = pd.melt(df_intake, id_vars='sex', value_vars=[var], var_name='variable', value_name='value')

    # Perform t-test
    t_stat, p_value = ttest_ind(df_intake[df_intake['sex'] == 'male'][var],
                                df_intake[df_intake['sex'] == 'female'][var], nan_policy='omit')

    # Plot by Sex
    sns.boxplot(data=melted_df, x='sex', y='value', ax=axs[i//3, i%3], palette='pastel')
    sns.stripplot(data=melted_df, x='sex', y='value', ax=axs[i//3, i%3], color='black', jitter=True, alpha=0.6)
    axs[i//3, i%3].set_title(f'{get_label(var)}\nt-stat: {t_stat:.2f}, p-value: {p_value:.2f}')
    axs[i//3, i%3].set_xlabel('Sex')
    axs[i//3, i%3].set_ylabel(get_label(var))

# Leave the last subplot blank
axs[2, 2].axis('off')

# Add panel label for Baseline (T1)
fig.text(0.05, 0.95, 'A.', fontsize=22, fontweight='bold', ha='center', va='center')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# Set up the figure and axes for follow-up variables
fig, axs = plt.subplots(3, 3, figsize=(18, 16))
fig.suptitle('Follow-up (T2) Intake Measures by Sex', fontsize=20)

for i, var in enumerate(followup_vars):
    # Prepare data for plotting
    melted_df = pd.melt(df_intake, id_vars='sex', value_vars=[var], var_name='variable', value_name='value')

    # Perform t-test
    t_stat, p_value = ttest_ind(df_intake[df_intake['sex'] == 'male'][var],
                                df_intake[df_intake['sex'] == 'female'][var], nan_policy='omit')

    # Plot by Sex
    sns.boxplot(data=melted_df, x='sex', y='value', ax=axs[i//3, i%3], palette='pastel')
    sns.stripplot(data=melted_df, x='sex', y='value', ax=axs[i//3, i%3], color='black', jitter=True, alpha=0.6)
    axs[i//3, i%3].set_title(f'{get_label(var)}\nt-stat: {t_stat:.2f}, p-value: {p_value:.2f}')
    axs[i//3, i%3].set_xlabel('Sex')
    axs[i//3, i%3].set_ylabel(get_label(var))

# Leave the last subplot blank
axs[2, 2].axis('off')

# Add panel label for Follow-up (T2)
fig.text(0.05, 0.95, 'B.', fontsize=22, fontweight='bold', ha='center', va='center')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


## sex differences in FMI

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# Assuming df_intake is the DataFrame containing the intake data
# Copy the dataframe for intake
df_intake = df.copy()

# Assuming 'sex' is a column in your DataFrame that represents gender information
# Map sex to labels if not already done
# Replace 'sex' with the actual column name if it's different
if 'sex' in df_intake:
    df_intake['sex'] = df_intake['sex'].map({0: 'male', 1: 'female'})

# Set up the figure and axes for the plots
fig, axs = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Differences in Baseline and Follow-up FMI by Sex', fontsize=20)

# Plot 1: Baseline FMI (v1_FMI) by Sex without showing boxplot fliers
baseline_t_stat, baseline_p_value = ttest_ind(
    df_intake[df_intake['sex'] == 'male']['v1_FMI'],
    df_intake[df_intake['sex'] == 'female']['v1_FMI'],
    nan_policy='omit'
)
sns.boxplot(data=df_intake, x='sex', y='v1_FMI', ax=axs[0], palette='pastel', showfliers=False)
sns.stripplot(data=df_intake, x='sex', y='v1_FMI', ax=axs[0], color='black', jitter=True, alpha=0.6)
axs[0].set_title(f'Baseline FMI\nt-stat: {baseline_t_stat:.2f}, p-value: {baseline_p_value:.2e}', fontsize=16)
axs[0].set_xlabel('Sex', fontsize=14)
axs[0].set_ylabel('Baseline FMI', fontsize=14)

# Plot 2: Follow-up FMI (v7_FMI) by Sex without showing boxplot fliers
followup_t_stat, followup_p_value = ttest_ind(
    df_intake[df_intake['sex'] == 'male']['v7_FMI'],
    df_intake[df_intake['sex'] == 'female']['v7_FMI'],
    nan_policy='omit'
)
sns.boxplot(data=df_intake, x='sex', y='v7_FMI', ax=axs[1], palette='pastel', showfliers=False)
sns.stripplot(data=df_intake, x='sex', y='v7_FMI', ax=axs[1], color='black', jitter=True, alpha=0.6)
axs[1].set_title(f'Follow-up FMI\nt-stat: {followup_t_stat:.2f}, p-value: {followup_p_value:.2e}', fontsize=16)
axs[1].set_xlabel('Sex', fontsize=14)
axs[1].set_ylabel('Follow-up FMI', fontsize=14)
# Add panel label
fig.text(0.05, 0.95, 'A.', fontsize=22, fontweight='bold', ha='center', va='center')
# Adjust layout and show the plot
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


## sex differences - change in FMI

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# Assuming df_intake is the DataFrame containing the intake data
# Copy the dataframe for intake
df_intake = df.copy()

# Assuming 'sex' is a column in your DataFrame that represents gender information
# Map sex to labels if not already done
# Replace 'sex' with the actual column name if it's different
if 'sex' in df_intake:
    df_intake['sex'] = df_intake['sex'].map({0: 'male', 1: 'female'})

# Calculate change in FMI
df_intake['change_FMI'] = df_intake['v7_FMI'] - df_intake['v1_FMI']

# Perform T-test for the change in FMI between boys and girls
boys_change_FMI = df_intake[df_intake['sex'] == 'male']['change_FMI']
girls_change_FMI = df_intake[df_intake['sex'] == 'female']['change_FMI']
t_stat, p_value = ttest_ind(boys_change_FMI, girls_change_FMI)

# Set up the figure for the plot
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_title(f'Change in FMI\nT-test p-value: {p_value:.3f}', fontsize=20)

# Plot: Change in FMI by Sex without showing boxplot fliers
sns.boxplot(data=df_intake, x='sex', y='change_FMI', palette='pastel', showfliers=False, ax=ax)
sns.stripplot(data=df_intake, x='sex', y='change_FMI', color='black', jitter=True, alpha=0.6, ax=ax)
ax.set_xlabel('Sex', fontsize=16)
ax.set_ylabel('Change in FMI', fontsize=16)

# Increase font size of tick labels
ax.tick_params(axis='both', which='major', labelsize=14)

# Add panel label
fig.text(0.05, 0.95, 'B.', fontsize=22, fontweight='bold', ha='center', va='center')

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

## sex differences in BMI-z at T1 and T2

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# Assuming df_intake is the DataFrame containing the intake data
# Copy the dataframe for intake
df_intake = df.copy()

# Assuming 'sex' is a column in your DataFrame that represents gender information
# Map sex to labels if not already done
# Replace 'sex' with the actual column name if it's different
if 'sex' in df_intake:
    df_intake['sex'] = df_intake['sex'].map({0: 'male', 1: 'female'})

# Perform T-test for baseline BMI-z
baseline_t_stat, baseline_p_value = ttest_ind(
    df_intake[df_intake['sex'] == 'male']['bmi_z'],
    df_intake[df_intake['sex'] == 'female']['bmi_z'],
    nan_policy='omit'
)

# Perform T-test for follow-up BMI-z
followup_t_stat, followup_p_value = ttest_ind(
    df_intake[df_intake['sex'] == 'male']['v7_bmi_z'],
    df_intake[df_intake['sex'] == 'female']['v7_bmi_z'],
    nan_policy='omit'
)

# Set up the figure and axes for the plots
fig, axs = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Differences in Baseline and Follow-up BMI-z by Sex', fontsize=20)

# Plot 1: Baseline BMI-z (bmi_z) by Sex without showing boxplot fliers
sns.boxplot(data=df_intake, x='sex', y='bmi_z', ax=axs[0], palette='pastel', showfliers=False)
sns.stripplot(data=df_intake, x='sex', y='bmi_z', ax=axs[0], color='black', jitter=True, alpha=0.6)
axs[0].set_title(f'Baseline BMI-z\nt-stat: {baseline_t_stat:.2f}, p-value: {baseline_p_value:.2e}', fontsize=16)
axs[0].set_xlabel('Sex', fontsize=14)
axs[0].set_ylabel('Baseline BMI-z', fontsize=14)

# Plot 2: Follow-up BMI-z (v7_bmi_z) by Sex without showing boxplot fliers
sns.boxplot(data=df_intake, x='sex', y='v7_bmi_z', ax=axs[1], palette='pastel', showfliers=False)
sns.stripplot(data=df_intake, x='sex', y='v7_bmi_z', ax=axs[1], color='black', jitter=True, alpha=0.6)
axs[1].set_title(f'Follow-up BMI-z\nt-stat: {followup_t_stat:.2f}, p-value: {followup_p_value:.2e}', fontsize=16)
axs[1].set_xlabel('Sex', fontsize=14)
axs[1].set_ylabel('Follow-up BMI-z', fontsize=14)

# Increase font size of tick labels
for ax in axs:
    ax.tick_params(axis='both', labelsize=12)

# Add panel label
fig.text(0.05, 0.95, 'C.', fontsize=22, fontweight='bold', ha='center', va='center')
# Adjust layout and show the plot
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


## Sex differences of METs at T1 and T2

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# Ensure Sex column is mapped correctly
df['sex'] = df['sex'].map({0: 'Boys', 1: 'Girls'})

# Plot the MET values for girls and boys
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='sex', y='Average_MET', palette='pastel', showfliers=False)
sns.stripplot(data=df, x='sex', y='Average_MET', color='black', jitter=True, alpha=0.6)
plt.title('METs by Sex')
plt.xlabel('sex')
plt.ylabel('Average_MET')
plt.show()

# Separate MET values by sex
boys_mets = df[df['sex'] == 'Boys']['Average_MET']
girls_mets = df[df['sex'] == 'Girls']['Average_MET']

# Perform t-test
t_stat, p_value = ttest_ind(boys_mets, girls_mets)

print(f'T-test results: t-statistic = {t_stat:.3f}, p-value = {p_value:.3f}')


# sex differences in Lean Mass Index

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# Sample data
import pandas as pd
import numpy as np

np.random.seed(0)
df = pd.DataFrame({
    'sex': np.random.choice([0, 1], 100),
    'v1_LMI': np.random.randn(100) * 2 + 20,
    'v7_LMI': np.random.randn(100) * 2 + 21
})

df_intake = df.copy()

# Map sex to labels if not already done
df_intake['sex'] = df_intake['sex'].map({0: 'male', 1: 'female'})

# Perform T-tests
t_stat_v1, p_value_v1 = ttest_ind(
    df_intake[df_intake['sex'] == 'male']['v1_LMI'],
    df_intake[df_intake['sex'] == 'female']['v1_LMI']
)

t_stat_v7, p_value_v7 = ttest_ind(
    df_intake[df_intake['sex'] == 'male']['v7_LMI'],
    df_intake[df_intake['sex'] == 'female']['v7_LMI']
)

# Set up the figure and axes for the plots
fig, axs = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Differences in Baseline and Follow-up Lean Mass Index by Sex', fontsize=16)

# Plot 1: Baseline LMI (v1_LMI) by Sex
sns.boxplot(data=df_intake, x='sex', y='v1_LMI', ax=axs[0], palette='pastel')
sns.stripplot(data=df_intake, x='sex', y='v1_LMI', ax=axs[0], color='black', jitter=True, alpha=0.6)
axs[0].set_title('Baseline LMI')
axs[0].set_xlabel('Sex')
axs[0].set_ylabel('Baseline LMI')

# Annotate T-value and p-value
axs[0].annotate(f'T-value: {t_stat_v1:.2f}\np-value: {p_value_v1:.3e}', xy=(0.5, 0.85), xycoords='axes fraction', ha='center', fontsize=12, color='black')

# Plot 2: Follow-up LMI (v7_LMI) by Sex
sns.boxplot(data=df_intake, x='sex', y='v7_LMI', ax=axs[1], palette='pastel')
sns.stripplot(data=df_intake, x='sex', y='v7_LMI', ax=axs[1], color='black', jitter=True, alpha=0.6)
axs[1].set_title('Follow-up LMI')
axs[1].set_xlabel('Sex')
axs[1].set_ylabel('Follow-up LMI')

# Annotate T-value and p-value
axs[1].annotate(f'T-value: {t_stat_v7:.2f}\np-value: {p_value_v7:.3e}', xy=(0.5, 0.85), xycoords='axes fraction', ha='center', fontsize=12, color='black')

# Adjust layout and show the plot
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


# Models by sex moderation

## meal or EAH_kcal:sex vs FMI

In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_eah_kcal'] = df_linear_reg['v1_eah_kcal'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_FMI ~ v1_FMI + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_eah_kcal * sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_eah_sweet_kcal'] = df_linear_reg['v1_eah_sweet_kcal'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_FMI ~ v1_FMI + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_eah_sweet_kcal * sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())



In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_eah_sav_kcal'] = df_linear_reg['v1_eah_sav_kcal'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_FMI ~ v1_FMI + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_eah_sav_kcal * sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_meal_kcal'] = df_linear_reg['v1_meal_kcal'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_FMI ~ v1_FMI + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_meal_kcal * sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())



## meal or EAH_g:sex vs FMI

In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_meal_g'] = df_linear_reg['v1_meal_g'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_FMI ~ v1_FMI + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_meal_g * sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_eah_g'] = df_linear_reg['v1_eah_g'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_FMI ~ v1_FMI + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_eah_g * sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_eah_sweet_g'] = df_linear_reg['v1_eah_sweet_g'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_FMI ~ v1_FMI + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_eah_sweet_g * sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_eah_sav_g'] = df_linear_reg['v1_eah_sav_g'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_FMI ~ v1_FMI + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_eah_sav_g * sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


## Meal or EAH_kcal : sex vs BMIz

In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_meal_kcal'] = df_linear_reg['v1_meal_kcal'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_bmi_z ~ bmi_z + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_meal_kcal* sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_eah_kcal'] = df_linear_reg['v1_eah_kcal'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_bmi_z ~ bmi_z + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_eah_kcal* sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_eah_sweet_kcal'] = df_linear_reg['v1_eah_sweet_kcal'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_bmi_z ~ bmi_z + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_eah_sweet_kcal* sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_eah_sav_kcal'] = df_linear_reg['v1_eah_sav_kcal'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_bmi_z ~ bmi_z + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_eah_sav_kcal* sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


## Meal_g or EAH_g : sex vs BMIz

In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_meal_g'] = df_linear_reg['v1_meal_g'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_bmi_z ~ bmi_z + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_meal_g* sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_eah_g'] = df_linear_reg['v1_eah_g'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_bmi_z ~ bmi_z + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_eah_g* sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_eah_sweet_g'] = df_linear_reg['v1_eah_sweet_g'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_bmi_z ~ bmi_z + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_eah_sweet_g* sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


In [ ]:
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt

# Create an interaction term between 'v1_eah_kcal' and 'sex'
df_linear_reg['interaction_eah_sav_g'] = df_linear_reg['v1_eah_sav_g'] * df_linear_reg['sex']

# Fit the regression model with the interaction term and other covariates
model = smf.ols('v7_bmi_z ~ bmi_z + risk_status_mom + income + parent_ed + v7_p_pds_imputed_2 + v7_p_pds_imputed_3 + age_diff+v1_eah_sav_g* sex', data=df_linear_reg).fit()

# Print the summary of the model to see the interaction effect
print(model.summary())


In [ ]:
import pandas as pd

# Data as provided
data = {
    'Intake Variable': [
        'Meal Kcal', 'EAH Kcal', 'EAH Sweet Kcal', 'EAH Savory Kcal',
        'Meal Grams', 'EAH Grams', 'EAH Sweet Grams', 'EAH Savory Grams'
    ],
    'Coefficient (SD)': [
        '0.1868 (0.111)', '0.1732 (0.102)', '0.1862 (0.102)', '-0.0002 (0.126)',
        '0.1592 (0.117)', '0.1585 (0.102)', '0.2279 (0.100)', '0.0061 (0.127)'
    ],
    'T-Value': [
        1.69, 1.70, 1.82, -0.00,
        1.36, 1.56, 2.27, 0.05
    ],
    'P-Value': [
        0.10, 0.10, 0.07, 1.00,
        0.18, 0.13, 0.03, 0.96
    ],
    'Conf. Interval [0.025, 0.975]': [
        '[-0.035, 0.408]', '[-0.031, 0.377]', '[-0.018, 0.391]', '[-0.251, 0.251]',
        '[-0.074, 0.393]', '[-0.045, 0.362]', '[0.027, 0.428]', '[-0.247, 0.259]'
    ]
}

# Create DataFrame
df = pd.DataFrame(data)

# Function to format Coefficient (SD)
def format_coeff_sd(value):
    coeff, sd = value.strip(')').split(' (')
    return f'{float(coeff):.2f} ({float(sd):.2f})'

# Apply the formatting function to Coefficient (SD) column
df['Coefficient (SD)'] = df['Coefficient (SD)'].apply(format_coeff_sd)

# Convert T-Value and P-Value to string with 2 decimal places
df['T-Value'] = df['T-Value'].map('{:.2f}'.format)
df['P-Value'] = df['P-Value'].map('{:.2f}'.format)

# Save DataFrame to CSV with float_format for numeric values
csv_file_name = "sex moderated models_FMI.csv"
df.to_csv(csv_file_name, index=False)

csv_file_name


In [ ]:
import pandas as pd

# Provided data as a string
data_str = """
v1_meal_kcal:sex       0.0201      0.087      0.231      0.818      -0.153       0.194
v1_eah_kcal:sex        0.0452      0.082      0.548      0.585      -0.120       0.210
v1_eah_sweet_kcal:sex     0.0576      0.083      0.695      0.490      -0.108       0.223
v1_eah_sav_kcal:sex    -0.0522      0.097     -0.539      0.592      -0.246       0.141
v1_meal_g:sex          0.0045      0.090      0.051      0.960      -0.175       0.184
v1_eah_g:sex           0.0712      0.080      0.893      0.375      -0.088       0.231
v1_eah_sweet_g:sex     0.0798      0.082      0.971      0.335      -0.085       0.244
v1_eah_sav_g:sex      -0.0494      0.097     -0.507      0.614      -0.244       0.145
"""

# Parse the data string
data_lines = [line.strip() for line in data_str.strip().split('\n')]
data_dict = {}
for line in data_lines:
    parts = line.split()
    key = parts[0].split(':')[0]
    coeff = float(parts[1])
    sd = float(parts[2])
    t_value = float(parts[3])
    p_value = float(parts[4])
    conf_interval = ' '.join(parts[5:])

    # Format the components
    coeff_sd_formatted = f'{coeff:.4f} ({sd:.3f})'
    t_value_formatted = f'{t_value:.3f}'
    p_value_formatted = f'{p_value:.3f}'

    # Store the formatted data
    data_dict[key] = [coeff_sd_formatted, t_value_formatted, p_value_formatted, conf_interval]

# Create DataFrame
df = pd.DataFrame.from_dict(data_dict, orient='index', columns=['Coefficient (SD)', 'T-Value', 'P-Value', 'Conf. Interval [0.025, 0.975]'])


# Save DataFrame to CSV with float_format for numeric values
csv_file_name = "sex moderated models_bmiz.csv"
df.to_csv(csv_file_name, index=False)

csv_file_name